# Diagnóstico de calidad de datos

**Actividad:** GA1-220501094-01-AA1-EV06 – Diagnosticar la calidad de datos: preparación, exploración y documentación.

**Proyecto:** Tecnologías y salarios de desarrolladores de software.

**Propósito:** identificar qué tecnologías se asocian con mejores salarios en el desarrollo de software, para orientar a quienes estudian o trabajan en el sector sobre qué habilidades aprender.

**Pregunta analítica:**

> ¿Qué lenguajes de programación y frameworks se asocian con un salario anual mayor entre desarrolladores profesionales de tiempo completo, y cuánto cambia esa diferencia según los años de experiencia y la región, entre 2022 y 2025?

Este cuaderno **no limpia** los datos: los carga, los inspecciona y documenta su calidad para decidir qué preparación necesitan antes del análisis.

## 1. Reconocimiento de los datasets

| # | DataFrame | Archivo (`data/raw/`) | Fuente | Qué contiene | Relación con la pregunta |
|---|---|---|---|---|---|
| 1 | `so_2022` | `so_survey_2022.csv` | Stack Overflow Developer Survey 2022 | Una fila por persona encuestada: perfil, empleo, experiencia, tecnologías usadas y salario | Salario (`ConvertedCompYearly`), lenguajes, frameworks, experiencia y país |
| 2 | `so_2023` | `so_survey_2023.csv` | Stack Overflow Developer Survey 2023 | Igual que el anterior, edición 2023 | Ídem |
| 3 | `so_2024` | `so_survey_2024.csv` | Stack Overflow Developer Survey 2024 | Igual que el anterior, edición 2024 | Ídem |
| 4 | `so_2025` | `so_survey_2025.csv` | Stack Overflow Developer Survey 2025 | Igual que el anterior, edición 2025 | Ídem |
| 5 | `pib_owid` | `pib_per_capita_owid.csv` | Our World in Data (dato del Banco Mundial) | PIB per cápita PPA, dólares internacionales constantes de 2021, formato largo (país-año) | Contexto económico de la región: comparar salarios entre países |
| 6 | `pib_bm` | `pib_per_capita_ppa_bm.csv` | Banco Mundial, indicador `NY.GDP.PCAP.PP.CD` | PIB per cápita PPA, dólares internacionales corrientes, formato ancho (un año por columna) | Ídem; segunda fuente para contrastar |

Los archivos `so_schema_<año>.csv` traen el texto de cada pregunta de la encuesta y se usan como diccionario de datos.

**Aspectos de calidad a revisar antes de usar los datos:**

1. Valores faltantes en salario, experiencia, tecnologías y país.
2. Tipos de datos: años de experiencia guardados como texto y columnas de selección múltiple.
3. Valores atípicos o imposibles en salario y experiencia.
4. Coherencia de categorías y nombres entre los cuatro años de la encuesta.
5. Coincidencia de nombres de país entre la encuesta y las fuentes de PIB.
6. Cobertura de años y presencia de regiones agregadas en las fuentes de PIB.

## 2. Preparación del entorno y carga de datos

Requisitos: entorno `.venv` activo con `requirements.txt` instalado y los datos descargados con:

```bash
python scripts/descargar_datos.py
```

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd

# pandas 3 guarda el texto con el tipo `str` y avisa que describe(include="object")
# dejará de incluirlo en una versión futura. La actividad pide ese comando, así que
# se usa tal cual y se silencia solo ese aviso.
warnings.filterwarnings("ignore", category=pd.errors.Pandas4Warning)

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

print("Python", sys.version.split()[0], "| pandas", pd.__version__)

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = RAIZ / "data" / "raw"
REPORTS = RAIZ / "reports"

archivos = [
    "so_survey_2022.csv", "so_survey_2023.csv", "so_survey_2024.csv", "so_survey_2025.csv",
    "pib_per_capita_owid.csv", "pib_per_capita_ppa_bm.csv",
]
faltan = [a for a in archivos if not (DATA / a).exists()]
if faltan:
    raise FileNotFoundError(f"Faltan {faltan}. Ejecuta: python scripts/descargar_datos.py")
for a in archivos:
    print(f"{a:28} {(DATA / a).stat().st_size / 1e6:8.1f} MB")

Python 3.14.7 | pandas 3.1.0
so_survey_2022.csv              108.8 MB
so_survey_2023.csv              158.6 MB
so_survey_2024.csv              159.5 MB
so_survey_2025.csv              140.9 MB
pib_per_capita_owid.csv           0.3 MB
pib_per_capita_ppa_bm.csv         0.2 MB


Cada dataset se carga en un DataFrame independiente. Todos son CSV, así que se usa `pd.read_csv`:

- En las encuestas se usa `low_memory=False` para que pandas lea el archivo completo antes de decidir el tipo de cada columna. Sin esto avisa de tipos mixtos.
- El CSV del Banco Mundial tiene **4 líneas de metadatos** antes del encabezado, por eso se usa `skiprows=4`.

In [2]:
so_2022 = pd.read_csv(DATA / "so_survey_2022.csv", low_memory=False)
so_2023 = pd.read_csv(DATA / "so_survey_2023.csv", low_memory=False)
so_2024 = pd.read_csv(DATA / "so_survey_2024.csv", low_memory=False)
so_2025 = pd.read_csv(DATA / "so_survey_2025.csv", low_memory=False)
pib_owid = pd.read_csv(DATA / "pib_per_capita_owid.csv")
pib_bm = pd.read_csv(DATA / "pib_per_capita_ppa_bm.csv", skiprows=4)

encuestas = {2022: so_2022, 2023: so_2023, 2024: so_2024, 2025: so_2025}
datasets = {
    "so_2022": so_2022, "so_2023": so_2023, "so_2024": so_2024, "so_2025": so_2025,
    "pib_owid": pib_owid, "pib_bm": pib_bm,
}

Verificación de la carga: cada DataFrame tiene filas y columnas, y en las encuestas `ResponseId` identifica a cada persona sin repetirse.

In [3]:
for nombre, df in datasets.items():
    memoria = df.memory_usage(deep=True).sum() / 1e6
    print(f"{nombre:9} filas={df.shape[0]:>6}  columnas={df.shape[1]:>3}  memoria={memoria:7.1f} MB")

for anio, df in encuestas.items():
    assert df["ResponseId"].is_unique, f"ResponseId repetido en {anio}"
print("\nCarga correcta: ResponseId es único en las cuatro encuestas.")

so_2022   filas= 73268  columnas= 79  memoria=  343.5 MB


so_2023   filas= 89184  columnas= 84  memoria=  460.2 MB


so_2024   filas= 65437  columnas=114  memoria=  438.8 MB


so_2025   filas= 49191  columnas=172  memoria=  375.2 MB
pib_owid  filas=  7445  columnas=  5  memoria=    1.4 MB
pib_bm    filas=   266  columnas= 71  memoria=    0.2 MB

Carga correcta: ResponseId es único en las cuatro encuestas.


## 3. Inspección de la estructura

Para cada DataFrame se ejecutan `head()`, `tail()`, `shape`, `columns`, `dtypes`, `info()`, `describe()` y `describe(include='object')`.

Las encuestas tienen entre 79 y 172 columnas. Con más de 100 columnas, `info()` solo muestra un resumen, así que se llama con `verbose=True, show_counts=True` para ver el conteo de no nulos de cada columna.

### 3.1 Encuesta 2022 (`so_2022`)

In [4]:
so_2022.head()

,ResponseId,MainBranch,Employment,RemoteWork,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,LearnCodeCoursesCert,YearsCode,YearsCodePro,DevType,OrgSize,PurchaseInfluence,BuyNewTool,...,Knowledge_6,Knowledge_7,Frequency_1,Frequency_2,Frequency_3,TimeSearching,TimeAnswering,Onboarding,ProfessionalTech,TrueFalse_1,TrueFalse_2,TrueFalse_3,SurveyLength,SurveyEase,ConvertedCompYearly
0,1,None of these,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,I am a developer by profession,"Employed, full-time",Fully remote,Hobby;Contribute to open-source projects,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Too long,Difficult,NaN
2,3,"I am not primarily a developer, but I write code sometim...","Employed, full-time","Hybrid (some remote, some in-person)",Hobby,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Books / Physical media;Friend or family member;Other onl...,Technical documentation;Blogs;Programming Games;Written ...,NaN,14,5,"Data scientist or machine learning specialist;Developer,...",20 to 99 employees,I have some influence,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Appropriate in length,Neither easy nor difficult,40205.0
3,4,I am a developer by profession,"Employed, full-time",Fully remote,I don’t code outside of work,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Books / Physical media;School (i.e., University, College...",NaN,NaN,20,17,"Developer, full-stack",100 to 499 employees,I have some influence,Other (please specify):,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Appropriate in length,Easy,215232.0
4,5,I am a developer by profession,"Employed, full-time","Hybrid (some remote, some in-person)",Hobby,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Other online resources (e.g., videos, blogs, forum);Scho...",Technical documentation;Blogs;Stack Overflow;Online book...,NaN,8,3,"Developer, front-end;Developer, full-stack;Developer, ba...",20 to 99 employees,I have some influence,Start a free trial;Visit developer communities like Stac...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Too long,Easy,NaN


In [5]:
so_2022.tail()

,ResponseId,MainBranch,Employment,RemoteWork,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,LearnCodeCoursesCert,YearsCode,YearsCodePro,DevType,OrgSize,PurchaseInfluence,BuyNewTool,...,Knowledge_6,Knowledge_7,Frequency_1,Frequency_2,Frequency_3,TimeSearching,TimeAnswering,Onboarding,ProfessionalTech,TrueFalse_1,TrueFalse_2,TrueFalse_3,SurveyLength,SurveyEase,ConvertedCompYearly
73263,73264,I am a developer by profession,"Employed, full-time",Fully remote,Freelance/contract work,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Books / Physical media;Other online resources (e.g., vid...",Technical documentation;Blogs;Written Tutorials;Stack Ov...,Udemy,8,5,"Developer, back-end",100 to 499 employees,I have some influence,Visit developer communities like Stack Overflow;Ask deve...,...,Strongly agree,Neither agree nor disagree,Never,Never,Never,30-60 minutes a day,Less than 15 minutes a day,Just right,DevOps function;Microservices;Developer portal or other ...,Yes,Yes,Yes,Too long,Easy,NaN
73264,73265,I am a developer by profession,"Employed, full-time",Full in-person,Hobby,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Other online resources (e.g., videos, blogs, forum);Scho...",Technical documentation;Blogs;Written Tutorials;Stack Ov...,Coursera;Udemy;Udacity,6,5,Data scientist or machine learning specialist,I don’t know,I have little or no influence,Other (please specify):;Ask developers I know/work with,...,Agree,Agree,1-2 times a week,6-10 times a week,10+ times a week,15-30 minutes a day,60-120 minutes a day,Very long,None of these,No,Yes,Yes,Too long,Easy,NaN
73265,73266,"I am not primarily a developer, but I write code sometim...","Employed, full-time","Hybrid (some remote, some in-person)",Hobby;School or academic work,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Books / Physical media;Other online resources (e.g., vid...",Technical documentation;Programming Games;Stack Overflow...,Udemy;Codecademy;Pluralsight;edX,42,33,"Developer, full-stack;Developer, desktop or enterprise a...",20 to 99 employees,I have a great deal of influence,Start a free trial;Ask developers I know/work with,...,Agree,Neither agree nor disagree,Never,Never,Never,30-60 minutes a day,60-120 minutes a day,Just right,None of these,No,No,No,Appropriate in length,Easy,NaN
73266,73267,I am a developer by profession,"Employed, full-time","Hybrid (some remote, some in-person)",Hobby,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;On the job training,NaN,NaN,50,31,"Developer, front-end;Developer, desktop or enterprise ap...",10 to 19 employees,I have a great deal of influence,Start a free trial;Visit developer communities like Stac...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Appropriate in length,Easy,NaN
73267,73268,"I used to be a developer by profession, but no longer am","Independent contractor, freelancer, or self-employed",Fully remote,Hobby;Contribute to open-source projects;Bootstrapping a...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Friend or family member;Other onl...,Technical documentation;Blogs;Programming Games;Written ...,Udemy;Pluralsight,16,5,"Developer, front-end;Engineer, data;Engineer, site relia...",NaN,NaN,Start a free trial;Visit developer communities like Stac...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Appropriate in length,Easy,NaN


In [6]:
print(so_2022.shape)
print(list(so_2022.columns))

(73268, 79)
['ResponseId', 'MainBranch', 'Employment', 'RemoteWork', 'CodingActivities', 'EdLevel', 'LearnCode', 'LearnCodeOnline', 'LearnCodeCoursesCert', 'YearsCode', 'YearsCodePro', 'DevType', 'OrgSize', 'PurchaseInfluence', 'BuyNewTool', 'Country', 'Currency', 'CompTotal', 'CompFreq', 'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 'MiscTechHaveWorkedWith', 'MiscTechWantToWorkWith', 'ToolsTechHaveWorkedWith', 'ToolsTechWantToWorkWith', 'NEWCollabToolsHaveWorkedWith', 'NEWCollabToolsWantToWorkWith', 'OpSysProfessional use', 'OpSysPersonal use', 'VersionControlSystem', 'VCInteraction', 'VCHostingPersonal use', 'VCHostingProfessional use', 'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith', 'OfficeStackSyncHaveWorkedWith', 'OfficeStackSyncWantToWorkWith', 'Blockchain', 'NEWSOSites', 'SOVisitFreq', 'SOAccount', '

In [7]:
print(so_2022.dtypes.value_counts())
so_2022.dtypes

str        73
float64     5
int64       1
Name: count, dtype: int64


ResponseId               int64
MainBranch                 str
Employment                 str
RemoteWork                 str
CodingActivities           str
                        ...   
TrueFalse_2                str
TrueFalse_3                str
SurveyLength               str
SurveyEase                 str
ConvertedCompYearly    float64
Length: 79, dtype: object

In [8]:
so_2022.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 73268 entries, 0 to 73267
Data columns (total 79 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   ResponseId                      73268 non-null  int64  
 1   MainBranch                      73268 non-null  str    
 2   Employment                      71709 non-null  str    
 3   RemoteWork                      58958 non-null  str    
 4   CodingActivities                58899 non-null  str    
 5   EdLevel                         71571 non-null  str    
 6   LearnCode                       71580 non-null  str    
 7   LearnCodeOnline                 50685 non-null  str    
 8   LearnCodeCoursesCert            29389 non-null  str    
 9   YearsCode                       71331 non-null  str    
 10  YearsCodePro                    51833 non-null  str    
 11  DevType                         61302 non-null  str    
 12  OrgSize                         51039 non-n

In [9]:
so_2022.describe()

,ResponseId,CompTotal,VCHostingPersonal use,VCHostingProfessional use,WorkExp,ConvertedCompYearly
count,73268.000000,3.842200e+04,0.0,0.0,36769.000000,3.807100e+04
mean,36634.500000,2.342434e+52,NaN,NaN,10.242378,1.707613e+05
std,21150.794099,4.591478e+54,NaN,NaN,8.706850,7.814132e+05
min,1.000000,0.000000e+00,NaN,NaN,0.000000,1.000000e+00
25%,18317.750000,3.000000e+04,NaN,NaN,4.000000,3.583200e+04
50%,36634.500000,7.750000e+04,NaN,NaN,8.000000,6.784500e+04
75%,54951.250000,1.540000e+05,NaN,NaN,15.000000,1.200000e+05
max,73268.000000,9.000000e+56,NaN,NaN,50.000000,5.000000e+07


In [10]:
so_2022.describe(include='object')

,MainBranch,Employment,RemoteWork,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,LearnCodeCoursesCert,YearsCode,YearsCodePro,DevType,OrgSize,PurchaseInfluence,BuyNewTool,Country,...,Knowledge_5,Knowledge_6,Knowledge_7,Frequency_1,Frequency_2,Frequency_3,TimeSearching,TimeAnswering,Onboarding,ProfessionalTech,TrueFalse_1,TrueFalse_2,TrueFalse_3,SurveyLength,SurveyEase
count,73268,71709,58958,58899,71571,71580,50685,29389,71331,51833,61302,51039,50969,67963,71771,...,35014,34991,34977,35371,35344,34515,36198,36022,35679,34906,35819,35715,35749,70444,70508
unique,6,103,3,63,9,737,7192,206,52,52,9984,10,3,125,180,...,5,5,5,5,5,5,5,5,5,155,2,2,2,3,3
top,I am a developer by profession,"Employed, full-time",Fully remote,Hobby,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","School (i.e., University, College, etc)",Technical documentation;Blogs;Written Tutorials;Stack Ov...,Udemy,10,3,"Developer, full-stack",20 to 99 employees,I have some influence,Start a free trial;Visit developer communities like Stac...,United States of America,...,Agree,Agree,Agree,1-2 times a week,1-2 times a week,1-2 times a week,30-60 minutes a day,15-30 minutes a day,Somewhat long,None of these,Yes,Yes,Yes,Appropriate in length,Easy
freq,53507,42962,25341,18118,30276,3669,715,5643,5217,4261,7142,10343,21991,11787,13543,...,18674,12916,13080,21689,16368,17666,13652,11871,12961,4658,23767,22716,28027,53883,47886


**Observaciones de `so_2022`:**

- Casi todas las columnas son texto (`str`). Solo 5 son `float64` y 1 es `int64` (`ResponseId`). Dos de las `float64`, `VCHostingPersonal use` y `VCHostingProfessional use`, están **100 % vacías**: pandas les asigna `float64` porque solo contienen `NaN`.
- `YearsCode` y `YearsCodePro` son años de experiencia, pero se leyeron como texto porque incluyen respuestas como `Less than 1 year` y `More than 50 years`.
- Las tecnologías (`LanguageHaveWorkedWith`, `WebframeHaveWorkedWith`, …) son **selección múltiple** guardada en una sola celda y separada por `;`.
- `DevType` también es de selección múltiple en 2022. Desde 2023 es de respuesta única.
- `describe()` de `ConvertedCompYearly` muestra un máximo de **50 millones de USD** y un mínimo de **1 USD**: hay valores atípicos.
- Es la única edición con `CompFreq` (frecuencia del pago).

### 3.2 Encuesta 2023 (`so_2023`)

In [11]:
so_2023.head()

,ResponseId,Q120,MainBranch,Age,Employment,RemoteWork,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,LearnCodeCoursesCert,YearsCode,YearsCodePro,DevType,OrgSize,...,Knowledge_4,Knowledge_5,Knowledge_6,Knowledge_7,Knowledge_8,Frequency_1,Frequency_2,Frequency_3,TimeSearching,TimeAnswering,ProfessionalTech,Industry,SurveyLength,SurveyEase,ConvertedCompYearly
0,1,I agree,None of these,18-24 years old,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,I agree,I am a developer by profession,25-34 years old,"Employed, full-time",Remote,Hobby;Contribute to open-source projects;Bootstrapping a...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Colleague;Friend or family member...,Formal documentation provided by the owner of the tech;B...,Other,18,9,"Senior Executive (C-Suite, VP, etc.)",2 to 9 employees,...,Agree,Agree,Agree,Agree,Strongly agree,1-2 times a week,10+ times a week,Never,15-30 minutes a day,15-30 minutes a day,DevOps function;Microservices;Automated testing;Observab...,"Information Services, IT, Software Development, or other...",Appropriate in length,Easy,285000.0
2,3,I agree,I am a developer by profession,45-54 years old,"Employed, full-time","Hybrid (some remote, some in-person)",Hobby;Professional development or self-paced learning fr...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Colleague;On the job training;Oth...,Formal documentation provided by the owner of the tech;B...,NaN,27,23,"Developer, back-end","5,000 to 9,999 employees",...,Agree,Agree,Agree,Agree,Agree,6-10 times a week,6-10 times a week,3-5 times a week,30-60 minutes a day,30-60 minutes a day,DevOps function;Microservices;Automated testing;Observab...,"Information Services, IT, Software Development, or other...",Appropriate in length,Easy,250000.0
3,4,I agree,I am a developer by profession,25-34 years old,"Employed, full-time","Hybrid (some remote, some in-person)",Hobby,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Colleague;Friend or family member;Other online resources...,Formal documentation provided by the owner of the tech;B...,NaN,12,7,"Developer, front-end",100 to 499 employees,...,Strongly agree,Agree,Neither agree nor disagree,Agree,Agree,1-2 times a week,10+ times a week,1-2 times a week,15-30 minutes a day,30-60 minutes a day,Automated testing;Continuous integration (CI) and (more ...,NaN,Appropriate in length,Easy,156000.0
4,5,I agree,I am a developer by profession,25-34 years old,"Employed, full-time;Independent contractor, freelancer, ...",Remote,Hobby;Contribute to open-source projects;Professional de...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Online Courses or Certification;O...,Formal documentation provided by the owner of the tech;B...,Other;Codecademy;edX,6,4,"Developer, full-stack",20 to 99 employees,...,Agree,Neither agree nor disagree,Agree,Strongly agree,Agree,1-2 times a week,1-2 times a week,3-5 times a week,60-120 minutes a day,30-60 minutes a day,Microservices;Automated testing;Observability tools;Cont...,Other,Appropriate in length,Neither easy nor difficult,23456.0


In [12]:
so_2023.tail()

,ResponseId,Q120,MainBranch,Age,Employment,RemoteWork,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,LearnCodeCoursesCert,YearsCode,YearsCodePro,DevType,OrgSize,...,Knowledge_4,Knowledge_5,Knowledge_6,Knowledge_7,Knowledge_8,Frequency_1,Frequency_2,Frequency_3,TimeSearching,TimeAnswering,ProfessionalTech,Industry,SurveyLength,SurveyEase,ConvertedCompYearly
89179,89180,I agree,I am a developer by profession,25-34 years old,"Employed, full-time;Independent contractor, freelancer, ...",Remote,Hobby;Bootstrapping a business;Freelance/contract work,"Associate degree (A.A., A.S., etc.)",Online Courses or Certification;Other online resources (...,Formal documentation provided by the owner of the tech;B...,Udemy,20,5,"Developer, front-end",100 to 499 employees,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Too long,Neither easy nor difficult,NaN
89180,89181,I agree,I am a developer by profession,18-24 years old,"Student, full-time;Employed, part-time","Hybrid (some remote, some in-person)",School or academic work,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Colleague;Online Courses or Certification;Other online r...,Formal documentation provided by the owner of the tech;R...,NaN,5,NaN,"Developer, back-end",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Too long,Easy,NaN
89181,89182,I agree,I code primarily as a hobby,Prefer not to say,I prefer not to say,NaN,NaN,Something else,Books / Physical media;Hackathons (virtual or in-person)...,NaN,Codecademy;Coursera,10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Too long,Neither easy nor difficult,NaN
89182,89183,I agree,I am a developer by profession,Under 18 years old,"Employed, part-time;Student, part-time","Hybrid (some remote, some in-person)",Hobby;School or academic work,"Secondary school (e.g. American high school, German Real...",Online Courses or Certification;Other online resources (...,Formal documentation provided by the owner of the tech;R...,Udemy,3,NaN,System administrator,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Appropriate in length,Neither easy nor difficult,NaN
89183,89184,I agree,I am a developer by profession,35-44 years old,"Employed, full-time","Hybrid (some remote, some in-person)",Hobby;Professional development or self-paced learning fr...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Colleague;Online Courses or Certification;Other online r...,Formal documentation provided by the owner of the tech;B...,Codecademy;Pluralsight;Coursera,17,12,"Developer, full-stack",100 to 499 employees,...,Strongly agree,Strongly agree,Agree,Neither agree nor disagree,Neither agree nor disagree,Never,1-2 times a week,1-2 times a week,60-120 minutes a day,30-60 minutes a day,DevOps function;Developer portal or other central places...,"Information Services, IT, Software Development, or other...",Appropriate in length,Easy,NaN


In [13]:
print(so_2023.shape)
print(list(so_2023.columns))

(89184, 84)
['ResponseId', 'Q120', 'MainBranch', 'Age', 'Employment', 'RemoteWork', 'CodingActivities', 'EdLevel', 'LearnCode', 'LearnCodeOnline', 'LearnCodeCoursesCert', 'YearsCode', 'YearsCodePro', 'DevType', 'OrgSize', 'PurchaseInfluence', 'TechList', 'BuyNewTool', 'Country', 'Currency', 'CompTotal', 'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 'MiscTechHaveWorkedWith', 'MiscTechWantToWorkWith', 'ToolsTechHaveWorkedWith', 'ToolsTechWantToWorkWith', 'NEWCollabToolsHaveWorkedWith', 'NEWCollabToolsWantToWorkWith', 'OpSysPersonal use', 'OpSysProfessional use', 'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith', 'OfficeStackSyncHaveWorkedWith', 'OfficeStackSyncWantToWorkWith', 'AISearchHaveWorkedWith', 'AISearchWantToWorkWith', 'AIDevHaveWorkedWith', 'AIDevWantToWorkWith', 'NEWSOSites', 'SOVisitFreq', 'SOAccount

In [14]:
print(so_2023.dtypes.value_counts())
so_2023.dtypes

str        80
float64     3
int64       1
Name: count, dtype: int64


ResponseId               int64
Q120                       str
MainBranch                 str
Age                        str
Employment                 str
                        ...   
ProfessionalTech           str
Industry                   str
SurveyLength               str
SurveyEase                 str
ConvertedCompYearly    float64
Length: 84, dtype: object

In [15]:
so_2023.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 89184 entries, 0 to 89183
Data columns (total 84 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   ResponseId                           89184 non-null  int64  
 1   Q120                                 89184 non-null  str    
 2   MainBranch                           89184 non-null  str    
 3   Age                                  89184 non-null  str    
 4   Employment                           87898 non-null  str    
 5   RemoteWork                           73810 non-null  str    
 6   CodingActivities                     73764 non-null  str    
 7   EdLevel                              87973 non-null  str    
 8   LearnCode                            87663 non-null  str    
 9   LearnCodeOnline                      70084 non-null  str    
 10  LearnCodeCoursesCert                 37076 non-null  str    
 11  YearsCode                            87

In [16]:
so_2023.describe()

,ResponseId,CompTotal,WorkExp,ConvertedCompYearly
count,89184.000000,4.822500e+04,43579.000000,4.801900e+04
mean,44592.500000,1.036807e+42,11.405126,1.031101e+05
std,25745.347541,2.276847e+44,9.051989,6.814188e+05
min,1.000000,0.000000e+00,0.000000,1.000000e+00
25%,22296.750000,6.300000e+04,5.000000,4.390700e+04
50%,44592.500000,1.150000e+05,9.000000,7.496300e+04
75%,66888.250000,2.300000e+05,16.000000,1.216410e+05
max,89184.000000,5.000000e+46,50.000000,7.435143e+07


In [17]:
so_2023.describe(include='object')

,Q120,MainBranch,Age,Employment,RemoteWork,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,LearnCodeCoursesCert,YearsCode,YearsCodePro,DevType,OrgSize,PurchaseInfluence,...,Knowledge_3,Knowledge_4,Knowledge_5,Knowledge_6,Knowledge_7,Knowledge_8,Frequency_1,Frequency_2,Frequency_3,TimeSearching,TimeAnswering,ProfessionalTech,Industry,SurveyLength,SurveyEase
count,89184,89184,89184,87898,73810,73764,87973,87663,70084,37076,87435,66136,76872,65043,64964,...,41798,41684,41527,41520,41467,41404,41916,41925,41054,42778,42629,41783,36774,86485,86554
unique,1,6,8,106,3,116,8,790,7940,210,52,52,33,10,3,...,5,5,5,5,5,5,5,5,5,5,5,284,12,3,3
top,I agree,I am a developer by profession,25-34 years old,"Employed, full-time","Hybrid (some remote, some in-person)",Hobby,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Other online resources (e.g., videos, blogs, forum)",Formal documentation provided by the owner of the tech;B...,Udemy,10,5,"Developer, full-stack",20 to 99 employees,I have some influence,...,Agree,Agree,Agree,Agree,Agree,Agree,1-2 times a week,1-2 times a week,1-2 times a week,30-60 minutes a day,15-30 minutes a day,None of these,"Information Services, IT, Software Development, or other...",Appropriate in length,Easy
freq,89184,67237,33247,53748,31131,15308,36706,4873,1182,7445,6521,4792,25735,13380,26805,...,16865,18978,22131,15171,15659,16430,25528,18930,21470,16338,13678,4871,18159,65962,54092


**Observaciones de `so_2023`:**

- Tiene la misma estructura que 2022, con 84 columnas y más preguntas sobre IA.
- `YearsCodePro` sigue siendo texto, con `Less than 1 year` y `More than 50 years`.
- `ConvertedCompYearly` llega a unos **74 millones de USD** de máximo y la desviación estándar supera por mucho a la mediana: la distribución está muy sesgada por valores extremos.
- `RemoteWork` cambia las etiquetas respecto a 2022 (`Remote` en vez de `Fully remote`).

### 3.3 Encuesta 2024 (`so_2024`)

In [18]:
so_2024.head()

,ResponseId,MainBranch,Age,Employment,RemoteWork,Check,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,TechDoc,YearsCode,YearsCodePro,DevType,OrgSize,...,ProfessionalQuestion,Industry,JobSatPoints_1,JobSatPoints_4,JobSatPoints_5,JobSatPoints_6,JobSatPoints_7,JobSatPoints_8,JobSatPoints_9,JobSatPoints_10,JobSatPoints_11,SurveyLength,SurveyEase,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,Under 18 years old,"Employed, full-time",Remote,Apples,Hobby,Primary/elementary school,Books / Physical media,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,I am a developer by profession,35-44 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other (please s...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Colleague;On the job training;Oth...,Technical documentation;Blogs;Books;Written Tutorials;St...,API document(s) and/or SDK document(s);User guides or RE...,20,17,"Developer, full-stack",NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
2,3,I am a developer by profession,45-54 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other (please s...,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Books / Physical media;Colleague;On the job training;Oth...,Technical documentation;Blogs;Books;Written Tutorials;St...,API document(s) and/or SDK document(s);User guides or RE...,37,27,Developer Experience,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Appropriate in length,Easy,NaN,NaN
3,4,I am learning to code,18-24 years old,"Student, full-time",NaN,Apples,NaN,Some college/university study without earning a degree,"Other online resources (e.g., videos, blogs, forum, onli...",Stack Overflow;How-to videos;Interactive tutorial,NaN,4,NaN,"Developer, full-stack",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Too long,Easy,NaN,NaN
4,5,I am a developer by profession,18-24 years old,"Student, full-time",NaN,Apples,NaN,"Secondary school (e.g. American high school, German Real...","Other online resources (e.g., videos, blogs, forum, onli...",Technical documentation;Blogs;Written Tutorials;Stack Ov...,API document(s) and/or SDK document(s);User guides or RE...,9,NaN,"Developer, full-stack",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Too short,Easy,NaN,NaN


In [19]:
so_2024.tail()

,ResponseId,MainBranch,Age,Employment,RemoteWork,Check,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,TechDoc,YearsCode,YearsCodePro,DevType,OrgSize,...,ProfessionalQuestion,Industry,JobSatPoints_1,JobSatPoints_4,JobSatPoints_5,JobSatPoints_6,JobSatPoints_7,JobSatPoints_8,JobSatPoints_9,JobSatPoints_10,JobSatPoints_11,SurveyLength,SurveyEase,ConvertedCompYearly,JobSat
65432,65433,I am a developer by profession,18-24 years old,"Employed, full-time",Remote,Apples,Hobby;School or academic work,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","On the job training;School (i.e., University, College, etc)",NaN,NaN,5,3,Blockchain,2 to 9 employees,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65433,65434,I am a developer by profession,25-34 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65434,65435,I am a developer by profession,25-34 years old,"Employed, full-time",In-person,Apples,Hobby,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Other online resources (e.g., videos, blogs, forum, onli...",Technical documentation;Stack Overflow;Social Media,API document(s) and/or SDK document(s);AI-powered search...,9,5,"Developer, mobile","1,000 to 4,999 employees",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65435,65436,I am a developer by profession,18-24 years old,"Employed, full-time","Hybrid (some remote, some in-person)",Apples,Hobby;Contribute to open-source projects;Professional de...,"Secondary school (e.g. American high school, German Real...","On the job training;Other online resources (e.g., videos...",Technical documentation;Blogs;Written Tutorials;Stack Ov...,API document(s) and/or SDK document(s);User guides or RE...,5,2,"Developer, back-end",20 to 99 employees,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
65436,65437,I code primarily as a hobby,18-24 years old,"Student, full-time",NaN,Apples,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
print(so_2024.shape)
print(list(so_2024.columns))

(65437, 114)
['ResponseId', 'MainBranch', 'Age', 'Employment', 'RemoteWork', 'Check', 'CodingActivities', 'EdLevel', 'LearnCode', 'LearnCodeOnline', 'TechDoc', 'YearsCode', 'YearsCodePro', 'DevType', 'OrgSize', 'PurchaseInfluence', 'BuyNewTool', 'BuildvsBuy', 'TechEndorse', 'Country', 'Currency', 'CompTotal', 'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'LanguageAdmired', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'DatabaseAdmired', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'PlatformAdmired', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 'WebframeAdmired', 'EmbeddedHaveWorkedWith', 'EmbeddedWantToWorkWith', 'EmbeddedAdmired', 'MiscTechHaveWorkedWith', 'MiscTechWantToWorkWith', 'MiscTechAdmired', 'ToolsTechHaveWorkedWith', 'ToolsTechWantToWorkWith', 'ToolsTechAdmired', 'NEWCollabToolsHaveWorkedWith', 'NEWCollabToolsWantToWorkWith', 'NEWCollabToolsAdmired', 'OpSysPersonal use', 'OpSysProfessional use', 'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWan

In [21]:
print(so_2024.dtypes.value_counts())
so_2024.dtypes

str        100
float64     13
int64        1
Name: count, dtype: int64


ResponseId               int64
MainBranch                 str
Age                        str
Employment                 str
RemoteWork                 str
                        ...   
JobSatPoints_11        float64
SurveyLength               str
SurveyEase                 str
ConvertedCompYearly    float64
JobSat                 float64
Length: 114, dtype: object

In [22]:
so_2024.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 65437 entries, 0 to 65436
Data columns (total 114 columns):
 #    Column                          Non-Null Count  Dtype  
---   ------                          --------------  -----  
 0    ResponseId                      65437 non-null  int64  
 1    MainBranch                      65437 non-null  str    
 2    Age                             65437 non-null  str    
 3    Employment                      65437 non-null  str    
 4    RemoteWork                      54806 non-null  str    
 5    Check                           65437 non-null  str    
 6    CodingActivities                54466 non-null  str    
 7    EdLevel                         60784 non-null  str    
 8    LearnCode                       60488 non-null  str    
 9    LearnCodeOnline                 49237 non-null  str    
 10   TechDoc                         40897 non-null  str    
 11   YearsCode                       59869 non-null  str    
 12   YearsCodePro               

In [23]:
so_2024.describe()

,ResponseId,CompTotal,WorkExp,JobSatPoints_1,JobSatPoints_4,JobSatPoints_5,JobSatPoints_6,JobSatPoints_7,JobSatPoints_8,JobSatPoints_9,JobSatPoints_10,JobSatPoints_11,ConvertedCompYearly,JobSat
count,65437.000000,3.374000e+04,29658.000000,29324.000000,29393.000000,29411.000000,29450.000000,29448.00000,29456.000000,29456.000000,29450.000000,29445.000000,2.343500e+04,29126.000000
mean,32719.000000,2.963841e+145,11.466957,18.581094,7.522140,10.060857,24.343232,22.96522,20.278165,16.169432,10.955713,9.953948,8.615529e+04,6.935041
std,18890.179119,5.444117e+147,9.168709,25.966221,18.422661,21.833836,27.089360,27.01774,26.108110,24.845032,22.906263,21.775652,1.867570e+05,2.088259
min,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,1.000000e+00,0.000000
25%,16360.000000,6.000000e+04,4.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,3.271200e+04,6.000000
50%,32719.000000,1.100000e+05,9.000000,10.000000,0.000000,0.000000,20.000000,15.00000,10.000000,5.000000,0.000000,0.000000,6.500000e+04,7.000000
75%,49078.000000,2.500000e+05,16.000000,22.000000,5.000000,10.000000,30.000000,30.00000,25.000000,20.000000,10.000000,10.000000,1.079715e+05,8.000000
max,65437.000000,1.000000e+150,50.000000,100.000000,100.000000,100.000000,100.000000,100.00000,100.000000,100.000000,100.000000,100.000000,1.625660e+07,10.000000


In [24]:
so_2024.describe(include='object')

,MainBranch,Age,Employment,RemoteWork,Check,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,TechDoc,YearsCode,YearsCodePro,DevType,OrgSize,PurchaseInfluence,...,Knowledge_7,Knowledge_8,Knowledge_9,Frequency_1,Frequency_2,Frequency_3,TimeSearching,TimeAnswering,Frustration,ProfessionalTech,ProfessionalCloud,ProfessionalQuestion,Industry,SurveyLength,SurveyEase
count,65437,65437,65437,54806,65437,54466,60784,60488,49237,40897,59869,51610,59445,47480,47406,...,27778,27758,27635,28369,28364,27710,28911,28844,28251,27764,28491,28807,28858,56182,56238
unique,5,8,110,3,1,118,8,418,10853,113,52,52,34,10,3,...,5,5,5,5,5,5,5,5,954,514,3,9,15,3,3
top,I am a developer by profession,25-34 years old,"Employed, full-time","Hybrid (some remote, some in-person)",Apples,Hobby,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Other online resources (e.g., videos, blogs, forum, onli...",Technical documentation;Blogs;Written Tutorials;Stack Ov...,API document(s) and/or SDK document(s);User guides or RE...,10,2,"Developer, full-stack",20 to 99 employees,I have some influence,...,Agree,Agree,Agree,1-2 times a week,1-2 times a week,1-2 times a week,30-60 minutes a day,15-30 minutes a day,None of these,None of these,Hybrid (on-prem and cloud),Traditional public search engine,Software Development,Appropriate in length,Easy
freq,50207,23911,39041,23015,65437,9993,24942,3674,603,6947,4561,4168,18260,9754,19197,...,10279,10759,6657,16989,12578,14426,10951,9341,2364,2834,13230,15813,11918,38767,30071


**Observaciones de `so_2024`:**

- Tiene 114 columnas, 13 de ellas numéricas. Además de salario y experiencia, se agregan las de satisfacción laboral (`JobSatPoints_*`, `JobSat`).
- **`ConvertedCompYearly` tiene la mayor proporción de nulos de las cuatro ediciones** (ver sección 4.1).
- `Currency` mezcla dos formatos (`USD\tUnited States dollar` y `EUR European Euro`).
- Algunos frameworks se escriben distinto que en 2022 (`ASP.NET CORE`, `React` y `AngularJS` en vez de `ASP.NET Core `, `React.js` y `Angular.js`).

### 3.4 Encuesta 2025 (`so_2025`)

In [25]:
so_2025.head()

,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,AILearnHow,YearsCode,DevType,OrgSize,ICorPM,...,AIAgentChallengesStrongly agree,AIAgentChallengesSomewhat agree,AIAgentChallengesStrongly disagree,AIAgentKnowledge,AIAgentKnowWrite,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8.0,"Yes, I am not new to coding but am learning new coding t...",Online Courses or Certification (includes all media type...,"Yes, I learned how to use AI-enabled tools for my person...",AI CodeGen tools or AI-enabled apps,14.0,"Developer, mobile",20 to 99 employees,People manager,...,The cost of using certain AI agent platforms is a barrie...,NaN,NaN,NaN,NaN,Vertex AI,NaN,NaN,NaN,ChatGPT,NaN,When I don’t trust AI’s answers,"Troubleshooting, profiling, debugging",61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,NaN,2.0,"Yes, I am not new to coding but am learning new coding t...",Online Courses or Certification (includes all media type...,"Yes, I learned how to use AI-enabled tools for my person...",AI CodeGen tools or AI-enabled apps,10.0,"Developer, back-end",500 to 999 employees,Individual contributor,...,I am concerned about the accuracy of the information pro...,Integrating AI agents with my existing tools and workflo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to fully und...,All skills. AI is a flop.,104413.0,9.0
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-employed",None of the above,10.0,"Yes, I am not new to coding but am learning new coding t...",Online Courses or Certification (includes all media type...,"Yes, I learned how to use AI-enabled tools for my person...",AI CodeGen tools or AI-enabled apps;Technical documentat...,12.0,"Developer, front-end",NaN,NaN,...,NaN,I am concerned about the accuracy of the information pro...,NaN,Redis,NaN,NaN,NaN,NaN,NaN,ChatGPT;Claude Code;GitHub Copilot;Google Gemini,NaN,When I don’t trust AI’s answers;When I want to fully und...,"Understand how things actually work, problem solving, an...",53061.0,8.0
3,4,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,None of the above,4.0,"Yes, I am not new to coding but am learning new coding t...","Other online resources (e.g. standard search, forum, onl...","Yes, I learned how to use AI-enabled tools for my person...",AI CodeGen tools or AI-enabled apps;Videos (not associat...,5.0,"Developer, back-end","10,000 or more employees",Individual contributor,...,I am concerned about the accuracy of the information pro...,Integrating AI agents with my existing tools and workflo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ChatGPT;Claude Code,NaN,When I don’t trust AI’s answers;When I want to fully und...,NaN,36197.0,6.0
4,5,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-employed","Caring for dependents (children, elderly, etc.)",21.0,"No, I am not new to coding and did not learn new coding ...",NaN,"Yes, I learned how to use AI-enabled tools for my person...",Technical documentation (is generated for/by the tool or...,22.0,Engineering manager,NaN,NaN,...,I am concerned about the accuracy of the information pro...,It takes significant time and effort to learn how to use...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers,"critical thinking, the skill to define the task correctly",60000.0,7.0


In [26]:
so_2025.tail()

,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,AILearnHow,YearsCode,DevType,OrgSize,ICorPM,...,AIAgentChallengesStrongly agree,AIAgentChallengesSomewhat agree,AIAgentChallengesStrongly disagree,AIAgentKnowledge,AIAgentKnowWrite,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
49186,49187,I code primarily as a hobby,18-24 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49187,49188,I am a developer by profession,45-54 years old,Some college/university study without earning a degree,Employed,"Caring for dependents (children, elderly, etc.)",25.0,"Yes, I am not new to coding but am learning new coding t...",Online Courses or Certification (includes all media types),"Yes, I learned how to use AI-enabled tools required for ...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49188,49189,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,None of the above,17.0,"Yes, I am not new to coding but am learning new coding t...",Online Courses or Certification (includes all media type...,"Yes, I learned how to use AI-enabled tools for my person...",AI CodeGen tools or AI-enabled apps;Videos (not associat...,31.0,"Architect, software or solutions","10,000 or more employees",People manager,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.0
49189,49190,I am a developer by profession,25-34 years old,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)","Independent contractor, freelancer, or self-employed",None of the above,2.0,"Yes, I am not new to coding but am learning new coding t...",Videos (not associated with specific online course or ce...,"Yes, I learned how to use AI-enabled tools for my person...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49190,49191,"I am not primarily a developer, but I write code sometim...",18-24 years old,Some college/university study without earning a degree,Student,Attending school (full-time),NaN,"Yes, I am not new to coding but am learning new coding t...",Online Courses or Certification (includes all media type...,"Yes, I learned how to use AI-enabled tools for my person...",AI CodeGen tools or AI-enabled apps,6.0,Student,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
print(so_2025.shape)
print(list(so_2025.columns))

(49191, 172)
['ResponseId', 'MainBranch', 'Age', 'EdLevel', 'Employment', 'EmploymentAddl', 'WorkExp', 'LearnCodeChoose', 'LearnCode', 'LearnCodeAI', 'AILearnHow', 'YearsCode', 'DevType', 'OrgSize', 'ICorPM', 'RemoteWork', 'PurchaseInfluence', 'TechEndorseIntro', 'TechEndorse_1', 'TechEndorse_2', 'TechEndorse_3', 'TechEndorse_4', 'TechEndorse_5', 'TechEndorse_6', 'TechEndorse_7', 'TechEndorse_8', 'TechEndorse_9', 'TechEndorse_13', 'TechEndorse_13_TEXT', 'TechOppose_1', 'TechOppose_2', 'TechOppose_3', 'TechOppose_5', 'TechOppose_7', 'TechOppose_9', 'TechOppose_11', 'TechOppose_13', 'TechOppose_16', 'TechOppose_15', 'TechOppose_15_TEXT', 'Industry', 'JobSatPoints_1', 'JobSatPoints_2', 'JobSatPoints_3', 'JobSatPoints_4', 'JobSatPoints_5', 'JobSatPoints_6', 'JobSatPoints_7', 'JobSatPoints_8', 'JobSatPoints_9', 'JobSatPoints_10', 'JobSatPoints_11', 'JobSatPoints_13', 'JobSatPoints_14', 'JobSatPoints_15', 'JobSatPoints_16', 'JobSatPoints_15_TEXT', 'AIThreat', 'NewRole', 'ToolCountWork', 'Too

In [28]:
print(so_2025.dtypes.value_counts())
so_2025.dtypes

str        119
float64     52
int64        1
Name: count, dtype: int64


ResponseId               int64
MainBranch                 str
Age                        str
EdLevel                    str
Employment                 str
                        ...   
AIAgentExtWrite            str
AIHuman                    str
AIOpen                     str
ConvertedCompYearly    float64
JobSat                 float64
Length: 172, dtype: object

In [29]:
so_2025.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Data columns (total 172 columns):
 #    Column                                    Non-Null Count  Dtype  
---   ------                                    --------------  -----  
 0    ResponseId                                49191 non-null  int64  
 1    MainBranch                                49191 non-null  str    
 2    Age                                       49191 non-null  str    
 3    EdLevel                                   48149 non-null  str    
 4    Employment                                48339 non-null  str    
 5    EmploymentAddl                            44875 non-null  str    
 6    WorkExp                                   42893 non-null  float64
 7    LearnCodeChoose                           46858 non-null  str    
 8    LearnCode                                 33556 non-null  str    
 9    LearnCodeAI                               45201 non-null  str    
 10   AILearnHow                     

In [30]:
so_2025.describe()

,ResponseId,WorkExp,YearsCode,TechEndorse_1,TechEndorse_2,TechEndorse_3,TechEndorse_4,TechEndorse_5,TechEndorse_6,TechEndorse_7,TechEndorse_8,TechEndorse_9,TechEndorse_13,TechOppose_1,TechOppose_2,...,ToolCountWork,ToolCountPersonal,CompTotal,SO_Actions_1,SO_Actions_16,SO_Actions_3,SO_Actions_4,SO_Actions_5,SO_Actions_6,SO_Actions_9,SO_Actions_7,SO_Actions_10,SO_Actions_15,ConvertedCompYearly,JobSat
count,49191.000000,42893.000000,43042.000000,35975.000000,35975.000000,35975.000000,35975.000000,35975.000000,35975.000000,35975.000000,35975.000000,35975.000000,35975.000000,34259.000000,34259.000000,...,27611.000000,25582.000000,2.486600e+04,26260.000000,24340.000000,26260.000000,26260.000000,26260.000000,26260.000000,26260.000000,26260.000000,26260.000000,26260.000000,2.394700e+04,26670.000000
mean,24596.000000,13.367403,16.570861,7.867352,4.104211,4.110271,5.678193,4.119388,5.225990,6.477387,4.370841,4.431661,9.854315,8.307948,4.524067,...,17.734852,8.545579,2.234198e+70,4.907845,3.339770,5.718355,4.561767,4.790861,5.199657,5.676314,4.984653,7.099505,10.079284,1.017615e+05,7.201950
std,14200.362883,10.800117,11.787610,2.397432,2.275821,2.329536,2.398084,2.437945,2.801045,2.331468,2.357058,2.620901,1.975692,2.202164,2.282867,...,269.810134,44.664738,3.523096e+72,2.682204,2.411404,2.628016,3.070548,2.643177,2.563562,2.310659,2.490095,2.469394,1.940928,4.617569e+05,1.997245
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000e+00,0.000000
25%,12298.500000,5.000000,8.000000,7.000000,2.000000,2.000000,4.000000,2.000000,3.000000,5.000000,2.000000,2.000000,10.000000,8.000000,3.000000,...,4.000000,3.000000,5.500000e+04,3.000000,1.000000,3.000000,1.000000,3.000000,3.000000,4.000000,3.000000,6.000000,10.000000,3.817100e+04,6.000000
50%,24596.000000,10.000000,14.000000,9.000000,4.000000,4.000000,6.000000,4.000000,5.000000,7.000000,4.000000,4.000000,10.000000,9.000000,4.000000,...,6.000000,5.000000,1.050000e+05,5.000000,2.000000,6.000000,4.000000,5.000000,5.000000,6.000000,5.000000,8.000000,10.000000,7.532000e+04,8.000000
75%,36893.500000,20.000000,24.000000,9.000000,6.000000,6.000000,8.000000,6.000000,8.000000,8.000000,6.000000,7.000000,10.000000,9.000000,6.000000,...,10.000000,7.000000,2.400000e+05,7.000000,5.000000,8.000000,7.000000,7.000000,7.000000,7.000000,7.000000,9.000000,10.000000,1.205960e+05,8.000000
max,49191.000000,100.000000,100.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,15.000000,15.000000,...,10000.000000,1000.000000,5.555556e+74,15.000000,10.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,5.000000e+07,10.000000


In [31]:
so_2025.describe(include='object')

,MainBranch,Age,EdLevel,Employment,EmploymentAddl,LearnCodeChoose,LearnCode,LearnCodeAI,AILearnHow,DevType,OrgSize,ICorPM,RemoteWork,PurchaseInfluence,TechEndorseIntro,...,AIAgentChallengesNeutral,AIAgentChallengesSomewhat disagree,AIAgentChallengesStrongly agree,AIAgentChallengesSomewhat agree,AIAgentChallengesStrongly disagree,AIAgentKnowledge,AIAgentKnowWrite,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen
count,49191,49191,48149,48339,44875,46858,33556,45201,28257,43680,34178,33243,33780,37437,37556,...,21295,12125,23108,21947,9314,3401,766,3760,478,2692,264,8332,859,29194,22540
unique,6,7,8,6,171,3,2767,5,1677,32,9,2,5,5,3,...,126,89,123,124,83,528,561,620,332,314,195,1311,421,574,20138
top,I am a developer by profession,25-34 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,None of the above,"Yes, I am not new to coding but am learning new coding t...","Other online resources (e.g. standard search, forum, onl...","Yes, I learned how to use AI-enabled tools required for ...",AI CodeGen tools or AI-enabled apps,"Developer, full-stack",20 to 99 employees,Individual contributor,Remote,No,Work,...,Integrating AI agents with my existing tools and workflo...,My company's IT and/or InfoSec teams have strict rules t...,I am concerned about the accuracy of the information pro...,I am concerned about the accuracy of the information pro...,My company's IT and/or InfoSec teams have strict rules t...,GitHub MCP Server,Copilot,Ollama,n8n,Grafana + Prometheus,no,ChatGPT;GitHub Copilot,Cursor,When I don’t trust AI’s answers;When I want to fully und...,Problem solving
freq,37467,16519,20278,33750,18613,32362,632,16403,2555,12351,6688,28341,10931,19697,25947,...,2334,2484,4433,1758,5502,657,15,796,28,588,13,669,106,3050,133


**Observaciones de `so_2025`:**

- Tiene 172 columnas, 52 de ellas numéricas, por las nuevas preguntas sobre agentes de IA.
- **Desaparece `YearsCodePro`**. La experiencia profesional queda solo en `WorkExp`, que ya es numérica, con un máximo de 100 años, un valor imposible.
- **`Employment` cambia de categorías**: ahora es `Employed`, sin distinguir *full-time* de *part-time*.
- `LanguageHaveWorkedWith` y `WebframeHaveWorkedWith` tienen muchos más nulos que en años anteriores.
- `Country` tiene 28 % de nulos, frente a 1–10 % en los años anteriores.

### 3.5 PIB per cápita – Our World in Data (`pib_owid`)

In [32]:
pib_owid.head()

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2000,1617.8264,Asia
1,Afghanistan,AFG,2001,1454.1108,Asia
2,Afghanistan,AFG,2002,1774.3087,Asia
3,Afghanistan,AFG,2003,1815.9282,Asia
4,Afghanistan,AFG,2004,1776.9182,Asia


In [33]:
pib_owid.tail()

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
7440,Zimbabwe,ZWE,2021,4827.1094,Africa
7441,Zimbabwe,ZWE,2022,5036.7830,Africa
7442,Zimbabwe,ZWE,2023,5218.0454,Africa
7443,Zimbabwe,ZWE,2024,5211.8975,Africa
7444,Zimbabwe,ZWE,2025,5529.2330,Africa


In [34]:
print(pib_owid.shape)
print(list(pib_owid.columns))

(7445, 5)
['entity', 'code', 'year', 'ny_gdp_pcap_pp_kd', 'owid_region']


In [35]:
print(pib_owid.dtypes.value_counts())
pib_owid.dtypes

str        3
int64      1
float64    1
Name: count, dtype: int64


entity                   str
code                     str
year                   int64
ny_gdp_pcap_pp_kd    float64
owid_region              str
dtype: object

In [36]:
pib_owid.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 7445 entries, 0 to 7444
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   entity             7445 non-null   str    
 1   code               7445 non-null   str    
 2   year               7445 non-null   int64  
 3   ny_gdp_pcap_pp_kd  7445 non-null   float64
 4   owid_region        6977 non-null   str    
dtypes: float64(1), int64(1), str(3)
memory usage: 290.9 KB


In [37]:
pib_owid.describe()

,year,ny_gdp_pcap_pp_kd
count,7445.000000,7445.000000
mean,2007.685426,22020.575246
std,10.333317,24019.749943
min,1990.000000,510.822800
25%,1999.000000,4597.220000
50%,2008.000000,12807.166000
75%,2017.000000,32722.330000
max,2025.000000,174569.520000


In [38]:
pib_owid.describe(include='object')

,entity,code,owid_region
count,7445,7445,6977
unique,212,212,6
top,Albania,ALB,Africa
freq,36,36,1848


**Observaciones de `pib_owid`:**

- Está en formato **largo** y ordenado: una fila por entidad y año, con `year` de tipo `int64` y el PIB como `float64`.
- Cubre de 1990 a 2025. No hay nulos en el PIB.
- `owid_region` es nula en las filas que **no son países** sino agregados (`World`, `High-income countries`, regiones del BM, …).
- La unidad es **dólares internacionales constantes de 2021** (`ny_gdp_pcap_pp_kd`).

### 3.6 PIB per cápita PPA – Banco Mundial (`pib_bm`)

In [39]:
pib_bm.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,34095.647171,35901.645961,35657.287197,35972.859662,36117.535262,37524.914920,39287.059713,38554.716783,28733.604511,35207.170909,41863.273996,47521.876547,51712.153136,NaN,NaN
1,Africa Eastern and Southern,AFE,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3244.317034,3373.428766,3513.894491,3539.428265,3639.496476,3843.028379,3727.710785,3804.533834,3707.033612,4022.277659,4359.591407,4496.744898,4635.125049,4840.265200,NaN
2,Afghanistan,AFG,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1988.429304,2133.241271,2224.490748,2284.075848,2213.181441,2335.795862,2432.276701,2583.485332,2561.981761,2144.166570,2122.659362,2203.537619,2236.199073,NaN,NaN
3,Africa Western and Central,AFW,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4666.588518,4867.130763,5121.421991,5063.905375,4972.623859,5035.762391,5188.368000,5491.687935,5423.594405,5702.134886,6232.323014,6538.812038,6840.333882,7184.664483,NaN
4,Angola,AGO,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,8246.360812,8504.095321,9017.690182,8067.485847,7882.513705,8125.706394,8370.228758,8621.659308,7782.151534,8720.187687,9373.408309,9549.793139,9963.590209,10251.345706,NaN


In [40]:
pib_bm.tail()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
261,Kosovo,XKX,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7897.566928,8090.759393,8200.474205,8714.919664,9157.046307,9380.940073,10018.738708,10861.053175,10532.505469,12362.421018,13948.685789,15817.432939,17890.360099,19263.326880,NaN
262,"Yemen, Rep.",YEM,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3004.920914,3164.321542,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
263,South Africa,ZAF,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12986.797668,13369.438204,13358.941719,13397.523481,13519.323510,13738.438585,13347.363483,13361.485129,12671.086244,13681.954814,14746.710370,15212.260960,15478.645679,15905.679442,NaN
264,Zambia,ZMB,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3323.633122,3477.281788,3441.314976,3337.473069,3313.351216,3391.627685,3442.273072,3361.397503,3157.300093,3503.034914,3840.115826,4080.355442,4221.013971,4379.261310,NaN
265,Zimbabwe,ZWE,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3472.485720,3783.946337,3903.427977,4045.898647,4275.449351,10756.445442,3992.890719,4165.922427,4178.509676,4827.109602,5395.024529,5795.990470,5932.870220,6470.416132,NaN


In [41]:
print(pib_bm.shape)
print(list(pib_bm.columns))

(266, 71)
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Unnamed: 70']


In [42]:
print(pib_bm.dtypes.value_counts())
pib_bm.dtypes

float64    67
str         4
Name: count, dtype: int64


Country Name          str
Country Code          str
Indicator Name        str
Indicator Code        str
1960              float64
                   ...   
2022              float64
2023              float64
2024              float64
2025              float64
Unnamed: 70       float64
Length: 71, dtype: object

In [43]:
pib_bm.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 71 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    266 non-null    str    
 1   Country Code    266 non-null    str    
 2   Indicator Name  266 non-null    str    
 3   Indicator Code  266 non-null    str    
 4   1960            0 non-null      float64
 5   1961            0 non-null      float64
 6   1962            0 non-null      float64
 7   1963            0 non-null      float64
 8   1964            0 non-null      float64
 9   1965            0 non-null      float64
 10  1966            0 non-null      float64
 11  1967            0 non-null      float64
 12  1968            0 non-null      float64
 13  1969            0 non-null      float64
 14  1970            0 non-null      float64
 15  1971            0 non-null      float64
 16  1972            0 non-null      float64
 17  1973            0 non-null      float64
 18  1

In [44]:
pib_bm.describe()

,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
count,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,247.000000,248.000000,247.000000,247.000000,246.000000,247.000000,247.000000,247.000000,247.000000,247.000000,247.000000,245.000000,243.000000,233.000000,0.0
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,19020.706121,19443.117739,19861.738754,19530.760647,20137.019419,21089.951457,22146.112415,23235.262459,21978.827901,24670.583161,27642.901926,29269.167561,30219.957009,30421.967058,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,22267.829681,22373.910641,21918.942174,19964.063908,20169.368498,21262.479813,22438.623988,23179.285360,21407.437343,24751.488508,27779.835889,28881.436080,29781.059921,30220.167389,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,637.247080,687.370968,698.963739,722.032898,764.282222,791.393495,823.013102,868.423732,946.413218,1032.382741,1101.235796,1147.424405,1193.035285,1248.054250,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4031.472289,4142.054301,4418.222881,4597.698725,4804.249179,5110.512257,5558.887454,5896.000719,5845.332799,6168.127082,6682.411118,7243.146763,7678.297058,7871.091291,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,10988.502614,11673.835571,12476.534436,12386.600335,13203.848923,13640.259799,14752.332318,15683.125924,14706.317304,16200.899460,18032.415056,18986.376295,19903.184889,20181.424283,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,24844.483335,25100.124692,26362.465102,26968.349419,28614.164774,30133.818477,32174.778327,34102.368606,32167.714509,37232.038183,41696.341235,45864.612543,46404.499062,45263.649851,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,180939.425775,169203.261058,148389.108964,108656.763905,113365.176082,120774.963081,129457.998600,126029.622371,122066.739147,136772.443753,146919.145398,152596.126692,156079.368561,163353.990260,NaN


In [45]:
pib_bm.describe(include='object')

,Country Name,Country Code,Indicator Name,Indicator Code
count,266,266,266,266
unique,266,266,1,1
top,Aruba,ABW,"GDP per capita, PPP (current international $)",NY.GDP.PCAP.PP.CD
freq,1,1,266,266


**Observaciones de `pib_bm`:**

- Está en formato **ancho**: una columna por año (`1960` … `2025`). Para unirlo con la encuesta habrá que pasarlo a formato largo con `melt`.
- Los nombres de columna de los años son texto y la última columna, `Unnamed: 70`, está **vacía**: es un artefacto de la coma final de cada línea del CSV.
- `Indicator Name` e `Indicator Code` tienen un único valor: son columnas constantes.
- Mezcla **266 entidades** entre países y agregados regionales o por nivel de ingreso.
- La unidad es **dólares internacionales corrientes** (`NY.GDP.PCAP.PP.CD`), distinta de la de OWID.

### 3.7 Tabla de diagnóstico estructural

La tabla se construye con los resultados de pandas. Se guarda también en `reports/tabla_diagnostico_estructural.csv`.

In [46]:
observaciones = {
    "so_2022": "YearsCode/YearsCodePro como texto; tecnologías y DevType en selección múltiple con ';'; salario con extremos (1 a 50 M USD); 2 columnas VCHosting 100 % vacías.",
    "so_2023": "Misma estructura que 2022; YearsCodePro sigue como texto; salario máximo de ~74 M USD.",
    "so_2024": "Salario con la mayor proporción de nulos; Currency con dos formatos; cambios de nombre en frameworks.",
    "so_2025": "Sin YearsCodePro (solo WorkExp); Employment sin distinguir tiempo completo; más nulos en tecnologías y país.",
    "pib_owid": "Formato largo país-año sin nulos en el PIB; incluye agregados (owid_region nula); USD PPA constantes 2021.",
    "pib_bm": "Formato ancho (un año por columna); columna vacía 'Unnamed: 70'; incluye agregados; USD PPA corrientes.",
}

filas = []
for nombre, df in datasets.items():
    tipos = df.dtypes.astype(str).value_counts()
    filas.append({
        "Dataset": nombre,
        "Filas": df.shape[0],
        "Columnas": df.shape[1],
        "Tipos de datos": ", ".join(f"{t}: {n}" for t, n in tipos.items()),
        "% celdas nulas": round(df.isna().mean().mean() * 100, 1),
        "Observaciones": observaciones[nombre],
    })

tabla_estructural = pd.DataFrame(filas)
REPORTS.mkdir(exist_ok=True)
tabla_estructural.to_csv(REPORTS / "tabla_diagnostico_estructural.csv", index=False, encoding="utf-8")
tabla_estructural

,Dataset,Filas,Columnas,Tipos de datos,% celdas nulas,Observaciones
0,so_2022,73268,79,"str: 73, float64: 5, int64: 1",28.4,YearsCode/YearsCodePro como texto; tecnologías y DevType...
1,so_2023,89184,84,"str: 80, float64: 3, int64: 1",33.6,Misma estructura que 2022; YearsCodePro sigue como texto...
2,so_2024,65437,114,"str: 100, float64: 13, int64: 1",38.8,Salario con la mayor proporción de nulos; Currency con d...
3,so_2025,49191,172,"str: 119, float64: 52, int64: 1",52.4,Sin YearsCodePro (solo WorkExp); Employment sin distingu...
4,pib_owid,7445,5,"str: 3, int64: 1, float64: 1",1.3,Formato largo país-año sin nulos en el PIB; incluye agre...
5,pib_bm,266,71,"float64: 67, str: 4",48.0,Formato ancho (un año por columna); columna vacía 'Unnam...


## 4. Diagnóstico inicial de calidad

Para analizar la calidad se define la **población de interés** de la pregunta: desarrolladores profesionales (`MainBranch == "I am a developer by profession"`) con empleo de tiempo completo.

### 4.1 Valores faltantes

Las 10 columnas con más nulos de cada encuesta:

In [47]:
def resumen_nulos(df, top=10):
    nulos = df.isna().sum()
    return (
        pd.DataFrame({"nulos": nulos, "% nulos": (nulos / len(df) * 100).round(1)})
        .sort_values("% nulos", ascending=False)
        .head(top)
    )

for anio, df in encuestas.items():
    print(f"\n=== Encuesta {anio} ===")
    print(resumen_nulos(df).to_string())


=== Encuesta 2022 ===
                                nulos  % nulos
VCHostingProfessional use       73268    100.0
VCHostingPersonal use           73268    100.0
LearnCodeCoursesCert            43879     59.9
OfficeStackAsyncWantToWorkWith  41196     56.2
Frequency_3                     38753     52.9
ProfessionalTech                38362     52.4
Knowledge_2                     38295     52.3
Knowledge_7                     38291     52.3
Knowledge_6                     38277     52.2
Knowledge_5                     38254     52.2

=== Encuesta 2023 ===


                                     nulos  % nulos
AINextVery similar                   86563     97.1
AINextSomewhat similar               82946     93.0
AINextNeither different nor similar  82585     92.6
AINextVery different                 76523     85.8
AIDevWantToWorkWith                  69597     78.0
AIToolNot interested in Using        68115     76.4
AINextSomewhat different             65881     73.9
AIDevHaveWorkedWith                  63280     71.0
AIToolInterested in Using            56401     63.2
AIToolCurrently Using                53047     59.5

=== Encuesta 2024 ===
                               nulos  % nulos
AINextMuch less integrated     64289     98.2
AINextLess integrated          63082     96.4
AINextNo change                52939     80.9
AINextMuch more integrated     51999     79.5
EmbeddedAdmired                48704     74.4
EmbeddedWantToWorkWith         47837     73.1
EmbeddedHaveWorkedWith         43223     66.1
ConvertedCompYearly            42002 

                      nulos  % nulos
AIAgentObsWrite       48927     99.5
SOTagsWant Entry      48761     99.1
SOTagsHaveEntry       48733     99.1
AIModelsWantEntry     48716     99.0
AIAgentOrchWrite      48713     99.0
JobSatPoints_15_TEXT  48527     98.7
AIAgentKnowWrite      48425     98.4
AIModelsHaveEntry     48415     98.4
SO_Actions_15_TEXT    48368     98.3
AIAgentExtWrite       48332     98.3


Porcentaje de nulos en las **variables clave** para la pregunta analítica, por año (`NaN` = la columna no existe ese año):

In [48]:
VARIABLES_CLAVE = [
    "MainBranch", "Employment", "Country", "YearsCodePro", "WorkExp",
    "LanguageHaveWorkedWith", "WebframeHaveWorkedWith", "ConvertedCompYearly",
]

nulos_clave = pd.DataFrame(
    {anio: [round(df[c].isna().mean() * 100, 1) if c in df.columns else None for c in VARIABLES_CLAVE]
     for anio, df in encuestas.items()},
    index=VARIABLES_CLAVE,
)
nulos_clave

,2022,2023,2024,2025
MainBranch,0.0,0.0,0.0,0.0
Employment,2.1,1.4,0.0,1.7
Country,2.0,1.4,9.9,28.0
YearsCodePro,29.3,25.8,21.1,NaN
WorkExp,49.8,51.1,54.7,12.8
LanguageHaveWorkedWith,3.1,2.3,8.7,35.6
WebframeHaveWorkedWith,26.9,24.9,31.0,53.3
ConvertedCompYearly,48.0,46.2,64.2,51.3


Muchos nulos del salario se explican por el diseño de la encuesta: la pregunta solo se muestra a quien tiene empleo. Por eso se mide también dentro de la población de interés:

In [49]:
def es_profesional(df):
    return df["MainBranch"].eq("I am a developer by profession")

def es_tiempo_completo(df, anio):
    # 2022-2024 distinguen "Employed, full-time"; 2025 solo tiene "Employed".
    if anio < 2025:
        return df["Employment"].str.contains("Employed, full-time", regex=False, na=False)
    return df["Employment"].eq("Employed")

filas = []
for anio, df in encuestas.items():
    pob = df[es_profesional(df) & es_tiempo_completo(df, anio)]
    filas.append({
        "Año": anio,
        "Respuestas": len(df),
        "Profesionales tiempo completo": len(pob),
        "% salario nulo (población)": round(pob["ConvertedCompYearly"].isna().mean() * 100, 1),
        "% lenguajes nulo (población)": round(pob["LanguageHaveWorkedWith"].isna().mean() * 100, 1),
        "% frameworks nulo (población)": round(pob["WebframeHaveWorkedWith"].isna().mean() * 100, 1),
        "% país nulo (población)": round(pob["Country"].isna().mean() * 100, 1),
        "Con salario y lenguajes": int((pob["ConvertedCompYearly"].notna() & pob["LanguageHaveWorkedWith"].notna()).sum()),
    })
poblacion = pd.DataFrame(filas).set_index("Año")
poblacion

,Respuestas,Profesionales tiempo completo,% salario nulo (población),% lenguajes nulo (población),% frameworks nulo (población),% país nulo (población),Con salario y lenguajes
Año,,,,,,,
2022,73268,43199,26.6,0.4,20.9,0.0,31647
2023,89184,53663,26.2,0.4,20.2,0.0,39529
2024,65437,39937,52.8,8.0,27.2,9.7,18785
2025,49191,28774,41.3,32.3,48.1,25.8,15787


También se pierde salario cuando la persona escribió un monto en `CompTotal` pero no se pudo convertir a USD (`ConvertedCompYearly` nulo):

In [50]:
pd.Series(
    {anio: int((df["CompTotal"].notna() & df["ConvertedCompYearly"].isna()).sum()) for anio, df in encuestas.items()},
    name="CompTotal informado sin conversión a USD",
)

2022      351
2023      206
2024    10305
2025      919
Name: CompTotal informado sin conversión a USD, dtype: int64

En 2024 son más de 10.000 casos, incluso en USD, y `CompTotal` tiene valores absurdos. Esto sugiere que la fuente descartó al convertir los montos poco creíbles:

In [51]:
sin_conversion_2024 = so_2024[so_2024["CompTotal"].notna() & so_2024["ConvertedCompYearly"].isna()]
print(sin_conversion_2024["Currency"].value_counts().head(5).to_string())
sin_conversion_2024["CompTotal"].describe()

Currency
EUR European Euro            3020
USD\tUnited States dollar    2710
GBP\tPound sterling           624
INR\tIndian rupee             503
CAD\tCanadian dollar          407


count     1.030500e+04
mean     9.704027e+145
std      9.850902e+147
min       0.000000e+00
25%       6.000000e+04
50%       1.100000e+05
75%       2.450000e+05
max      1.000000e+150
Name: CompTotal, dtype: float64

Valores faltantes en las fuentes de PIB:

In [52]:
print("pib_owid – nulos por columna:")
print(pib_owid.isna().sum().to_string())
print("\nEntidades de pib_owid sin owid_region (no son países):")
print(sorted(pib_owid.loc[pib_owid["owid_region"].isna(), "entity"].unique()))

anios_proyecto = ["2022", "2023", "2024", "2025"]
print("\npib_bm – entidades sin dato por año del proyecto:")
print(pib_bm[anios_proyecto].isna().sum().to_string())
print("\npib_owid – entidades con dato por año del proyecto:")
print(pib_owid[pib_owid["year"].between(2022, 2025)].groupby("year").size().to_string())

pib_owid – nulos por columna:
entity                 0
code                   0
year                   0
ny_gdp_pcap_pp_kd      0
owid_region          468

Entidades de pib_owid sin owid_region (no son países):
['East Asia and Pacific (WB)', 'Europe and Central Asia (WB)', 'European Union (27)', 'High-income countries', 'Latin America and Caribbean (WB)', 'Low-income countries', 'Lower-middle-income countries', 'Middle East, North Africa, Afghanistan and Pakistan (WB)', 'North America (WB)', 'South Asia (WB)', 'Sub-Saharan Africa (WB)', 'Upper-middle-income countries', 'World']

pib_bm – entidades sin dato por año del proyecto:
2022    19
2023    21
2024    23
2025    33

pib_owid – entidades con dato por año del proyecto:
year
2022    212
2023    210
2024    208
2025    198


**Interpretación:**

- El salario (`ConvertedCompYearly`) es la variable objetivo y tiene entre 46 % y 64 % de nulos en el total, sobre todo porque la pregunta no se muestra a estudiantes ni desempleados. Dentro de la población de interés el faltante baja a cerca de 26 % en 2022–2023, pero sigue alto en 2024 (≈53 %) y 2025 (≈41 %). En 2024 una parte se explica porque más de 10.000 montos de `CompTotal` no se convirtieron a USD. **Imputar el salario no es aceptable**, porque inventaría la variable que se quiere explicar. Las filas sin salario se excluyen.
- En 2025 los lenguajes y frameworks tienen muchos más nulos que en los años anteriores. Esto afecta la comparación temporal: menos casos por tecnología en 2025.
- En 2025 el país falta en un 28 % del total. Sin país no se puede analizar la región.
- El Banco Mundial no tiene dato 2025 para varias entidades. Para 2025 conviene usar el último año disponible de cada país o la fuente OWID.

### 4.2 Tipos de datos

**Años de experiencia guardados como texto.** Valores de `YearsCodePro` y `YearsCode` que no se pueden convertir a número:

In [53]:
def no_numericos(serie):
    convertida = pd.to_numeric(serie, errors="coerce")
    return serie[convertida.isna() & serie.notna()].value_counts()

for anio, df in encuestas.items():
    for col in ["YearsCodePro", "YearsCode"]:
        if col in df.columns:
            valores = no_numericos(df[col])
            print(f"{anio} {col:12} dtype={str(df[col].dtype):7} no numéricos={valores.to_dict()}")
        else:
            print(f"{anio} {col:12} NO EXISTE")
    print(f"{anio} {'WorkExp':12} dtype={str(df['WorkExp'].dtype)}")

2022 YearsCodePro dtype=str     no numéricos={'Less than 1 year': 1972, 'More than 50 years': 41}
2022 YearsCode    dtype=str     no numéricos={'Less than 1 year': 1413, 'More than 50 years': 172}
2022 WorkExp      dtype=float64
2023 YearsCodePro dtype=str     no numéricos={'Less than 1 year': 1836, 'More than 50 years': 83}
2023 YearsCode    dtype=str     no numéricos={'Less than 1 year': 968, 'More than 50 years': 353}
2023 WorkExp      dtype=float64


2024 YearsCodePro dtype=str     no numéricos={'Less than 1 year': 2856, 'More than 50 years': 50}


2024 YearsCode    dtype=str     no numéricos={'Less than 1 year': 569, 'More than 50 years': 254}
2024 WorkExp      dtype=float64
2025 YearsCodePro NO EXISTE
2025 YearsCode    dtype=float64 no numéricos={}
2025 WorkExp      dtype=float64


**Selección múltiple en una sola celda.** Proporción de respuestas con más de una tecnología y número de opciones distintas:

In [54]:
filas = []
for anio, df in encuestas.items():
    for col in ["LanguageHaveWorkedWith", "WebframeHaveWorkedWith", "DevType", "Employment"]:
        s = df[col].dropna()
        opciones = s.str.split(";").explode().str.strip().nunique()
        filas.append({"Año": anio, "Columna": col, "dtype": str(df[col].dtype),
                      "% con varias opciones": round(s.str.contains(";", regex=False).mean() * 100, 1),
                      "combinaciones distintas": s.nunique(), "opciones individuales": opciones})
pd.DataFrame(filas).pivot(index="Columna", columns="Año", values="% con varias opciones")

Año,2022,2023,2024,2025
Columna,,,,
DevType,60.8,0.0,0.0,0.0
Employment,15.9,14.5,16.9,0.0
LanguageHaveWorkedWith,93.6,93.8,93.8,96.3
WebframeHaveWorkedWith,77.2,78.5,78.3,83.4


In [55]:
so_2022["LanguageHaveWorkedWith"].head(3).tolist()

[nan, 'JavaScript;TypeScript', 'C#;C++;HTML/CSS;JavaScript;Python']

**Formato ancho y columnas sin información en `pib_bm`:**

In [56]:
print("Columnas de año como texto:", pib_bm.columns[4:8].tolist(), "…", pib_bm.columns[-3:].tolist())
print("Valores no nulos en 'Unnamed: 70':", pib_bm["Unnamed: 70"].notna().sum())
print("Valores distintos en 'Indicator Name':", pib_bm["Indicator Name"].unique().tolist())
print("Valores distintos en 'Indicator Code':", pib_bm["Indicator Code"].unique().tolist())

Columnas de año como texto: ['1960', '1961', '1962', '1963'] … ['2024', '2025', 'Unnamed: 70']
Valores no nulos en 'Unnamed: 70': 0
Valores distintos en 'Indicator Name': ['GDP per capita, PPP (current international $)']
Valores distintos en 'Indicator Code': ['NY.GDP.PCAP.PP.CD']


**Interpretación:**

- `YearsCodePro` (2022–2024) y `YearsCode` (2022–2024) son numéricos en su significado, pero pandas los leyó como texto por dos etiquetas: `Less than 1 year` y `More than 50 years`. Hay que reemplazarlas por un número (p. ej. 0,5 y 51) y convertir con `pd.to_numeric`. En 2025 `YearsCode` y `WorkExp` ya vienen numéricos: **el tipo de la misma variable cambia entre años**.
- Las tecnologías vienen como texto con varias opciones por celda. No sirven así para agrupar: hay que separarlas con `str.split(";")` y `explode()`, o crear columnas indicadoras (una por tecnología).
- `DevType` es de selección múltiple en 2022 y de respuesta única desde 2023: no es comparable entre años sin una regla.
- `Employment` también es de selección múltiple en 2022–2024 (p. ej. `Employed, full-time;Independent contractor…`).
- En `pib_bm` los años son columnas de texto. Habrá que pasarlo a formato largo con `melt` y convertir el año a entero. `Unnamed: 70` está vacía y `Indicator Name`/`Indicator Code` son constantes: se pueden eliminar.

### 4.3 Posibles inconsistencias

**Salarios atípicos o imposibles** (`ConvertedCompYearly`, USD anuales):

In [57]:
filas = []
for anio, df in encuestas.items():
    s = df["ConvertedCompYearly"].dropna()
    filas.append({
        "Año": anio, "n": len(s), "mínimo": s.min(), "p01": round(s.quantile(.01)),
        "mediana": round(s.median()), "media": round(s.mean()), "p99": round(s.quantile(.99)),
        "máximo": s.max(), "< 1.000 USD": int((s < 1_000).sum()), "> 1.000.000 USD": int((s > 1_000_000).sum()),
    })
salarios = pd.DataFrame(filas).set_index("Año")
salarios

,n,mínimo,p01,mediana,media,p99,máximo,< 1.000 USD,> 1.000.000 USD
Año,,,,,,,,,
2022,38071,1.0,1340,67845,170761,2161310,50000000.0,337,865
2023,48019,1.0,439,74963,103110,400000,74351432.0,684,60
2024,23435,1.0,208,65000,86155,393751,16256603.0,545,41
2025,23947,1.0,65,75320,101762,440856,50000000.0,724,46


**Experiencia incoherente:** años de experiencia profesional mayores que los años programando, o mayores de lo que permite la edad.

In [58]:
def a_numero(serie):
    return pd.to_numeric(
        serie.replace({"Less than 1 year": "0.5", "More than 50 years": "51"}), errors="coerce"
    )

EDAD_MAXIMA = {  # edad máxima de cada rango (para '65 years or older' se asume 100)
    "Under 18 years old": 17, "18-24 years old": 24, "25-34 years old": 34, "35-44 years old": 44,
    "45-54 years old": 54, "55-64 years old": 64, "65 years or older": 100,
}

filas = []
for anio, df in encuestas.items():
    exp = df["WorkExp"]
    edad_max = df["Age"].map(EDAD_MAXIMA)
    fila = {
        "Año": anio,
        "WorkExp máximo": exp.max(),
        "WorkExp > 50": int((exp > 50).sum()),
        "WorkExp > edad máxima - 14": int((exp > edad_max - 14).sum()),
    }
    if "YearsCodePro" in df.columns:
        pro, total = a_numero(df["YearsCodePro"]), a_numero(df["YearsCode"])
        fila["YearsCodePro > YearsCode"] = int((pro > total).sum())
    else:
        fila["YearsCodePro > YearsCode"] = None
    filas.append(fila)
experiencia = pd.DataFrame(filas).set_index("Año")
experiencia

,WorkExp máximo,WorkExp > 50,WorkExp > edad máxima - 14,YearsCodePro > YearsCode
Año,,,,
2022,50.0,0,111,544.0
2023,50.0,0,141,691.0
2024,50.0,0,107,504.0
2025,100.0,147,119,NaN


**Categorías que cambian entre años.** Las mismas preguntas usan etiquetas distintas:

In [59]:
for col in ["Employment", "RemoteWork", "Age"]:
    print(f"\n=== {col} ===")
    for anio, df in encuestas.items():
        vals = df[col].dropna().str.split(";").explode().unique()
        print(f"{anio}: {sorted(vals)[:8]}")


=== Employment ===
2022: ['Employed, full-time', 'Employed, part-time', 'I prefer not to say', 'Independent contractor, freelancer, or self-employed', 'Not employed, and not looking for work', 'Not employed, but looking for work', 'Retired', 'Student, full-time']
2023: ['Employed, full-time', 'Employed, part-time', 'I prefer not to say', 'Independent contractor, freelancer, or self-employed', 'Not employed, and not looking for work', 'Not employed, but looking for work', 'Retired', 'Student, full-time']
2024: ['Employed, full-time', 'Employed, part-time', 'I prefer not to say', 'Independent contractor, freelancer, or self-employed', 'Not employed, and not looking for work', 'Not employed, but looking for work', 'Retired', 'Student, full-time']
2025: ['Employed', 'I prefer not to say', 'Independent contractor, freelancer, or self-employed', 'Not employed', 'Retired', 'Student']

=== RemoteWork ===


2022: ['Full in-person', 'Fully remote', 'Hybrid (some remote, some in-person)']
2023: ['Hybrid (some remote, some in-person)', 'In-person', 'Remote']
2024: ['Hybrid (some remote, some in-person)', 'In-person', 'Remote']
2025: ['Hybrid (some in-person, leans heavy to flexibility)', 'Hybrid (some remote, leans heavy to in-person)', 'In-person', 'Remote', 'Your choice (very flexible, you can come in when you want or just as needed)']

=== Age ===
2022: ['18-24 years old', '25-34 years old', '35-44 years old', '45-54 years old', '55-64 years old', '65 years or older', 'Prefer not to say', 'Under 18 years old']
2023: ['18-24 years old', '25-34 years old', '35-44 years old', '45-54 years old', '55-64 years old', '65 years or older', 'Prefer not to say', 'Under 18 years old']


2024: ['18-24 years old', '25-34 years old', '35-44 years old', '45-54 years old', '55-64 years old', '65 years or older', 'Prefer not to say', 'Under 18 years old']
2025: ['18-24 years old', '25-34 years old', '35-44 years old', '45-54 years old', '55-64 years old', '65 years or older', 'Prefer not to say']


In [60]:
print("Formato de Currency (¿separa código y nombre con tabulador?):")
for anio, df in encuestas.items():
    c = df["Currency"].dropna()
    print(f"{anio}: con tabulador={int(c.str.contains(chr(9)).sum()):>6}  sin tabulador={int((~c.str.contains(chr(9))).sum()):>6}  ej: {c.iloc[0]!r}")

Formato de Currency (¿separa código y nombre con tabulador?):
2022: con tabulador= 38461  sin tabulador= 12803  ej: 'CAD\tCanadian dollar'
2023: con tabulador= 47490  sin tabulador= 17844  ej: 'USD\tUnited States dollar'
2024: con tabulador= 33810  sin tabulador= 12874  ej: 'PKR\tPakistani rupee'
2025: con tabulador=  8766  sin tabulador= 26671  ej: 'EUR European Euro'


**Nombres de tecnologías que cambian entre años.** Opciones de `LanguageHaveWorkedWith` y `WebframeHaveWorkedWith` que no están en los cuatro años:

In [61]:
def opciones(df, col):
    return set(df[col].dropna().str.split(";").explode())

for col in ["LanguageHaveWorkedWith", "WebframeHaveWorkedWith"]:
    por_anio = {anio: opciones(df, col) for anio, df in encuestas.items()}
    comunes = set.intersection(*por_anio.values())
    print(f"\n=== {col}: {len(comunes)} opciones en los 4 años ===")
    for anio, ops in por_anio.items():
        print(f"{anio} ({len(ops)} opciones) solo/no común: {sorted(ops - comunes)[:12]}")

print("\nVariantes de ASP.NET por año:")
for anio, df in encuestas.items():
    print(anio, sorted(o for o in opciones(df, "WebframeHaveWorkedWith") if "ASP" in o))


=== LanguageHaveWorkedWith: 31 opciones en los 4 años ===
2022 (42 opciones) solo/no común: ['APL', 'Bash/Shell', 'COBOL', 'Clojure', 'Crystal', 'Haskell', 'Julia', 'LISP', 'Objective-C', 'SAS', 'Solidity']
2023 (51 opciones) solo/no común: ['APL', 'Ada', 'Apex', 'Bash/Shell (all shells)', 'Clojure', 'Cobol', 'Crystal', 'Flow', 'GDScript', 'Haskell', 'Julia', 'Lisp']
2024 (49 opciones) solo/no común: ['Ada', 'Apex', 'Bash/Shell (all shells)', 'Clojure', 'Cobol', 'Crystal', 'GDScript', 'Haskell', 'Julia', 'Lisp', 'MicroPython', 'Nim']
2025 (42 opciones) solo/no común: ['Ada', 'Bash/Shell (all shells)', 'COBOL', 'GDScript', 'Gleam', 'Lisp', 'MicroPython', 'Mojo', 'Prolog', 'Visual Basic (.Net)', 'Zig']



=== WebframeHaveWorkedWith: 20 opciones en los 4 años ===
2022 (25 opciones) solo/no común: ['ASP.NET Core ', 'Angular.js', 'Gatsby', 'Play Framework', 'React.js']
2023 (34 opciones) solo/no común: ['ASP.NET CORE', 'AngularJS', 'CodeIgniter', 'Elm', 'Gatsby', 'Lit', 'NestJS', 'Play Framework', 'Qwik', 'React', 'Remix', 'Solid.js']
2024 (36 opciones) solo/no común: ['ASP.NET CORE', 'AngularJS', 'Astro', 'CodeIgniter', 'Elm', 'Gatsby', 'Htmx', 'NestJS', 'Play Framework', 'React', 'Remix', 'Solid.js']
2025 (28 opciones) solo/no común: ['ASP.NET Core', 'AngularJS', 'Astro', 'Axum', 'NestJS', 'React', 'Spring Boot', 'WordPress']

Variantes de ASP.NET por año:
2022 ['ASP.NET', 'ASP.NET Core ']


2023 ['ASP.NET', 'ASP.NET CORE']
2024 ['ASP.NET', 'ASP.NET CORE']
2025 ['ASP.NET', 'ASP.NET Core']


**Nombres de país que no coinciden con las fuentes de PIB.** Es necesario para unir salario y contexto económico:

In [62]:
paises_owid = set(pib_owid["entity"])
paises_bm = set(pib_bm["Country Name"])

filas = []
for anio, df in encuestas.items():
    pais = df["Country"].dropna()
    sin_owid = pais[~pais.isin(paises_owid)]
    filas.append({
        "Año": anio,
        "países en encuesta": pais.nunique(),
        "sin coincidencia OWID": sin_owid.nunique(),
        "sin coincidencia BM": pais[~pais.isin(paises_bm)].nunique(),
        "% respuestas sin coincidencia OWID": round(len(sin_owid) / len(pais) * 100, 1),
    })
display(pd.DataFrame(filas).set_index("Año"))

print("Países con más respuestas que no coinciden con OWID (2022):")
pais22 = so_2022["Country"].dropna()
pais22[~pais22.isin(paises_owid)].value_counts().head(10)

,países en encuesta,sin coincidencia OWID,sin coincidencia BM,% respuestas sin coincidencia OWID
Año,,,,
2022,180,28,32,30.1
2023,185,30,33,32.2
2024,185,30,33,29.7
2025,177,29,32,30.3


Países con más respuestas que no coinciden con OWID (2022):


Country
United States of America                                13543
United Kingdom of Great Britain and Northern Ireland     4190
Russian Federation                                       1229
Iran, Islamic Republic of...                              723
Czech Republic                                            692
Viet Nam                                                  317
Taiwan                                                    233
Hong Kong (S.A.R.)                                        184
Venezuela, Bolivarian Republic of...                      119
Republic of Korea                                          72
Name: count, dtype: int64

**Mezcla de países y agregados, y diferencia de unidades entre las dos fuentes de PIB:**

In [63]:
codigos_paises_owid = set(pib_owid.loc[pib_owid["owid_region"].notna(), "code"])
agregados_bm = pib_bm.loc[~pib_bm["Country Code"].isin(codigos_paises_owid), "Country Name"]
print(f"pib_bm: {len(pib_bm)} entidades, de las cuales {len(agregados_bm)} no son países de OWID (agregados u otros). Ej.:")
print(agregados_bm.head(12).tolist())

print("\nDuplicados: OWID (entity, year) =", pib_owid.duplicated(["entity", "year"]).sum(),
      "| BM Country Code =", pib_bm["Country Code"].duplicated().sum())

comparacion = (
    pib_owid[(pib_owid["year"] == 2023) & pib_owid["code"].isin(["COL", "USA", "IND", "DEU", "BRA"])]
    .set_index("code")[["entity", "ny_gdp_pcap_pp_kd"]]
    .join(pib_bm.set_index("Country Code")["2023"].rename("bm_pp_cd_2023"))
)
comparacion["diferencia %"] = ((comparacion["bm_pp_cd_2023"] / comparacion["ny_gdp_pcap_pp_kd"] - 1) * 100).round(1)
comparacion

pib_bm: 266 entidades, de las cuales 68 no son países de OWID (agregados u otros). Ej.:
['Africa Eastern and Southern', 'Africa Western and Central', 'Arab World', 'American Samoa', 'Central Europe and the Baltics', 'Channel Islands', 'Caribbean small states', 'Cuba', 'East Asia & Pacific (excluding high income)', 'Early-demographic dividend', 'East Asia & Pacific', 'Europe & Central Asia (excluding high income)']

Duplicados: OWID (entity, year) = 0 | BM Country Code = 0


,entity,ny_gdp_pcap_pp_kd,bm_pp_cd_2023,diferencia %
code,,,,
BRA,Brazil,19079.812,21193.072142,11.1
COL,Colombia,18406.625,21281.747521,15.6
DEU,Germany,63140.195,72262.223514,14.4
IND,India,8870.550,9853.043093,11.1
USA,United States,74351.670,82586.784771,11.1


**Interpretación:**

- **Salario:** hay salarios anuales de 1 USD y de decenas de millones de USD. Son errores de digitación, de moneda o de frecuencia (mensual escrito como anual). La media queda muy por encima de la mediana: hay que usar **mediana** y recortar extremos (p. ej. percentiles 1–99, o un rango plausible por país) antes de comparar tecnologías.
- **Experiencia:** en 2025 hay `WorkExp` de hasta 100 años, y en todos los años hay casos con más experiencia que la que permite la edad, o con más años profesionales que años programando. Son respuestas poco confiables que se deben marcar o excluir.
- **Categorías:** `Employment`, `RemoteWork` y `Age` cambian de etiquetas entre años. `Currency` mezcla dos formatos. Algunas tecnologías cambian de nombre: `ASP.NET Core ` (con espacio final, 2022), `ASP.NET CORE` (2023–2024) y `ASP.NET Core` (2025); `React.js`/`React`, `Angular.js`/`AngularJS`, `Bash/Shell`/`Bash/Shell (all shells)`, `COBOL`/`Cobol`, `LISP`/`Lisp`. Hay que **armonizar** las etiquetas con un diccionario y `str.strip()` antes de unir los años.
- **Países:** cerca de 30 nombres no coinciden con las fuentes de PIB, entre ellos los de más respuestas (`United States of America`, `United Kingdom of Great Britain and Northern Ireland`, `Russian Federation`, `Iran, Islamic Republic of...`). Sin una tabla de equivalencias se perdería cerca del 30 % de las respuestas al unir. También aparece `Nomadic`, que no es un país.
- **PIB:** las dos fuentes miden en **unidades distintas** (OWID: USD PPA constantes de 2021; BM: USD PPA corrientes), así que los valores difieren aunque sean del mismo país y año. No se deben mezclar en una misma columna. El BM incluye agregados regionales que hay que filtrar antes de unir.
- **Duplicados:** no hay filas ni `ResponseId` repetidos, ni pares país-año repetidos en las fuentes de PIB.

### 4.4 Limitaciones de los datos para la pregunta analítica

In [64]:
limitaciones = pd.DataFrame({
    "Profesionales tiempo completo": poblacion["Profesionales tiempo completo"],
    "Con salario y lenguajes": poblacion["Con salario y lenguajes"],
    "Existe YearsCodePro": ["YearsCodePro" in df.columns for df in encuestas.values()],
    "Employment distingue tiempo completo": [anio < 2025 for anio in encuestas],
})
limitaciones

,Profesionales tiempo completo,Con salario y lenguajes,Existe YearsCodePro,Employment distingue tiempo completo
Año,,,,
2022,43199,31647,True,True
2023,53663,39529,True,True
2024,39937,18785,True,True
2025,28774,15787,False,False


Concentración de respuestas por país:

In [65]:
filas = []
for anio, df in encuestas.items():
    conteo = df["Country"].value_counts()
    filas.append({"Año": anio, "top 5 países": ", ".join(conteo.index[:5]),
                  "% respuestas top 5": round(conteo.head(5).sum() / conteo.sum() * 100, 1),
                  "% EE. UU.": round(conteo.iloc[0] / conteo.sum() * 100, 1),
                  "países con < 30 respuestas": int((conteo < 30).sum())})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(filas).set_index("Año"))

,top 5 países,% respuestas top 5,% EE. UU.,países con < 30 respuestas
Año,,,,
2022,"United States of America, India, Germany, United Kingdom of Great Britain and Northern Ireland, Canada",44.9,18.9,73
2023,"United States of America, Germany, India, United Kingdom of Great Britain and Northern Ireland, Canada",46.2,21.2,76
2024,"United States of America, Germany, India, United Kingdom of Great Britain and Northern Ireland, Ukraine",44.4,18.8,85
2025,"United States of America, Germany, India, United Kingdom of Great Britain and Northern Ireland, France",45.9,20.4,99


1. **2025 no distingue tiempo completo.** `Employment` solo dice `Employed`. La población de 2025 incluye también medio tiempo, así que no es exactamente comparable con 2022–2024. Hay que declararlo como supuesto o restringir esa parte del análisis a 2022–2024.
2. **Experiencia no homogénea.** 2022–2024 tienen `YearsCodePro` (años programando profesionalmente) y `WorkExp` (experiencia laboral), pero 2025 solo `WorkExp`. Para comparar los cuatro años la variable común es `WorkExp`, aunque en 2022–2024 tiene alrededor de 50 % de nulos en el total.
3. **Muestra de Stack Overflow.** Responde quien usa la plataforma y quiere participar. No es una muestra aleatoria de todos los desarrolladores: EE. UU. aporta cerca de una de cada cinco respuestas, y los cinco países con más respuestas (EE. UU., Alemania, India, Reino Unido y un quinto que cambia según el año) reúnen más del 40 %. Más de 70 países tienen menos de 30 respuestas: no permiten conclusiones por país y conviene agrupar por región.
4. **Salario autodeclarado** y convertido a USD con tasas de cambio de cada año de la encuesta. No se ajusta por costo de vida, así que debe compararse con contexto (PIB PPA) o dentro de cada región.
5. **Asociación, no causalidad.** Saber un lenguaje no causa un mejor salario: puede reflejar el tipo de empresa, el país o el cargo.
6. **Opciones de tecnologías que cambian entre años.** Una tecnología que no se ofrecía en un año no se puede comparar en ese año.

### 4.5 Tabla de hallazgos y acciones propuestas

La evidencia de la tabla se calcula con las variables de las secciones anteriores, así que coincide con los resultados de pandas. Se guarda en `reports/tabla_hallazgos.csv`.

In [66]:
n = lambda x: f"{int(x):,}".replace(",", ".")
nulos_sal = nulos_clave.loc["ConvertedCompYearly"]

hallazgos = pd.DataFrame([
    ["Encuestas 2022–2025", "Valores faltantes en el salario (variable objetivo)",
     f"ConvertedCompYearly nulo: {nulos_sal[2022]} % (2022), {nulos_sal[2023]} %, {nulos_sal[2024]} %, {nulos_sal[2025]} % (2025). "
     f"En la población de interés 2024: {poblacion.loc[2024, '% salario nulo (población)']} %",
     "Menos casos por tecnología; 2024 queda con menos representación",
     "Filtrar la población de interés y excluir filas sin salario; no imputar la variable objetivo"],
    ["Encuestas 2022–2025", "Salarios atípicos o imposibles",
     f"Mínimo 1 USD en los 4 años; máximo {n(salarios['máximo'].max())} USD; "
     f"{n(salarios['> 1.000.000 USD'].sum())} respuestas > 1 M USD y {n(salarios['< 1.000 USD'].sum())} < 1.000 USD",
     "La media se distorsiona y las diferencias entre tecnologías pueden ser falsas",
     "Usar la mediana; recortar por percentiles 1–99 por año (o por país) y documentar el criterio"],
    ["Encuestas 2022–2024", "Años de experiencia como texto",
     "YearsCodePro y YearsCode de tipo str por 'Less than 1 year' y 'More than 50 years'",
     "No se pueden calcular rangos ni estadísticas de experiencia",
     "Reemplazar esas etiquetas (0,5 y 51) y convertir con pd.to_numeric"],
    ["Encuesta 2025", "Variable de experiencia profesional eliminada",
     "YearsCodePro no existe en 2025; solo WorkExp",
     "La experiencia no es comparable entre los 4 años con la misma variable",
     "Usar WorkExp como variable común y documentarlo"],
    ["Encuestas 2022–2025", "Experiencia incoherente",
     f"WorkExp máximo en 2025 = {n(experiencia.loc[2025, 'WorkExp máximo'])}; "
     f"{n(experiencia['WorkExp > edad máxima - 14'].sum())} casos con más experiencia que la que permite la edad; "
     f"{n(experiencia['YearsCodePro > YearsCode'].sum())} con más años profesionales que años programando (2022–2024)",
     "Sesga el análisis por nivel de experiencia",
     "Marcar y excluir respuestas incoherentes con reglas explícitas"],
    ["Encuesta 2025", "Employment sin distinción de tiempo completo",
     "Categorías 2025: 'Employed', 'Student', … en lugar de 'Employed, full-time'",
     "La población de 2025 no es la misma que pide la pregunta",
     "Usar 'Employed' como aproximación y declararlo como limitación, o limitar ese corte a 2022–2024"],
    ["Encuestas 2022–2025", "Selección múltiple en una sola celda",
     "LanguageHaveWorkedWith y WebframeHaveWorkedWith con varias opciones separadas por ';'",
     "No se puede agrupar el salario por tecnología directamente",
     "Separar con str.split(';') + explode() o crear columnas indicadoras por tecnología"],
    ["Encuestas 2022–2025", "Etiquetas que cambian entre años",
     "RemoteWork, Employment y Age con etiquetas distintas; Currency con 2 formatos; 'ASP.NET Core ' / 'ASP.NET CORE' / 'ASP.NET Core', 'React.js' / 'React'",
     "Al unir los años se crean categorías falsas o duplicadas",
     "Armonizar con un diccionario de equivalencias y str.strip()"],
    ["Encuesta 2025", "Más nulos en tecnologías y país",
     f"LanguageHaveWorkedWith nulo {nulos_clave.loc['LanguageHaveWorkedWith', 2025]} % y Country {nulos_clave.loc['Country', 2025]} % en 2025",
     "Menos casos por tecnología y región en el último año",
     "Excluir esas filas del análisis por tecnología o región y reportar el tamaño de muestra por año"],
    ["Encuestas vs. PIB", "Nombres de país no coinciden",
     "~30 países sin coincidencia con OWID (p. ej. 'United States of America', 'United Kingdom of Great Britain and Northern Ireland', 'Russian Federation')",
     "Al unir se perdería cerca del 30 % de las respuestas",
     "Crear una tabla de equivalencias de países o unir por código ISO3; excluir 'Nomadic'"],
    ["so_2022", "Columnas completamente vacías",
     f"VCHostingPersonal use y VCHostingProfessional use: {n(so_2022['VCHostingPersonal use'].isna().sum())} nulos de {n(len(so_2022))} filas (100 %)",
     "No aportan información y ocupan memoria",
     "Eliminarlas con drop(columns=...)"],
    ["pib_bm", "Formato ancho y columnas sin información",
     "Un año por columna (texto); 'Unnamed: 70' vacía; Indicator Name/Code constantes",
     "No se puede unir por año directamente",
     "Eliminar columnas vacías/constantes; pasar a formato largo con melt y convertir el año a int"],
    ["pib_owid", "Agregados mezclados con países y menor cobertura reciente",
     f"{pib_owid.loc[pib_owid['owid_region'].isna(), 'entity'].nunique()} entidades sin owid_region (World, regiones, niveles de ingreso); "
     f"{int((pib_owid['year'] == 2025).sum())} entidades con dato en 2025 vs. {int((pib_owid['year'] == 2022).sum())} en 2022",
     "Se podrían unir agregados como si fueran países; algunos países quedarían sin PIB en 2025",
     "Filtrar owid_region no nula; para 2025 usar el último año disponible por país"],
    ["pib_bm / pib_owid", "Agregados mezclados con países y unidades distintas",
     f"pib_bm: {len(agregados_bm)} entidades que no son países de OWID; OWID usa USD PPA constantes 2021 y BM USD PPA corrientes",
     "Se podrían unir regiones como si fueran países o mezclar unidades",
     "Filtrar solo países; usar una sola fuente (OWID) como principal y BM como contraste"],
    ["pib_bm", "PIB 2025 incompleto",
     f"{int(pib_bm['2025'].isna().sum())} entidades sin dato en 2025 (vs. {int(pib_bm['2022'].isna().sum())} en 2022)",
     "Algunos países no tendrían contexto económico en 2025",
     "Usar el último año disponible por país y documentarlo"],
], columns=["Dataset", "Problema identificado", "Evidencia encontrada", "Posible impacto", "Acción propuesta"])

hallazgos.to_csv(REPORTS / "tabla_hallazgos.csv", index=False, encoding="utf-8")
with pd.option_context("display.max_colwidth", None):
    display(hallazgos)

,Dataset,Problema identificado,Evidencia encontrada,Posible impacto,Acción propuesta
0,Encuestas 2022–2025,Valores faltantes en el salario (variable objetivo),"ConvertedCompYearly nulo: 48.0 % (2022), 46.2 %, 64.2 %, 51.3 % (2025). En la población de interés 2024: 52.8 %",Menos casos por tecnología; 2024 queda con menos representación,Filtrar la población de interés y excluir filas sin salario; no imputar la variable objetivo
1,Encuestas 2022–2025,Salarios atípicos o imposibles,Mínimo 1 USD en los 4 años; máximo 74.351.432 USD; 1.012 respuestas > 1 M USD y 2.290 < 1.000 USD,La media se distorsiona y las diferencias entre tecnologías pueden ser falsas,Usar la mediana; recortar por percentiles 1–99 por año (o por país) y documentar el criterio
2,Encuestas 2022–2024,Años de experiencia como texto,YearsCodePro y YearsCode de tipo str por 'Less than 1 year' y 'More than 50 years',No se pueden calcular rangos ni estadísticas de experiencia,"Reemplazar esas etiquetas (0,5 y 51) y convertir con pd.to_numeric"
3,Encuesta 2025,Variable de experiencia profesional eliminada,YearsCodePro no existe en 2025; solo WorkExp,La experiencia no es comparable entre los 4 años con la misma variable,Usar WorkExp como variable común y documentarlo
4,Encuestas 2022–2025,Experiencia incoherente,WorkExp máximo en 2025 = 100; 478 casos con más experiencia que la que permite la edad; 1.739 con más años profesionales que años programando (2022–2024),Sesga el análisis por nivel de experiencia,Marcar y excluir respuestas incoherentes con reglas explícitas
5,Encuesta 2025,Employment sin distinción de tiempo completo,"Categorías 2025: 'Employed', 'Student', … en lugar de 'Employed, full-time'",La población de 2025 no es la misma que pide la pregunta,"Usar 'Employed' como aproximación y declararlo como limitación, o limitar ese corte a 2022–2024"
6,Encuestas 2022–2025,Selección múltiple en una sola celda,LanguageHaveWorkedWith y WebframeHaveWorkedWith con varias opciones separadas por ';',No se puede agrupar el salario por tecnología directamente,Separar con str.split(';') + explode() o crear columnas indicadoras por tecnología
7,Encuestas 2022–2025,Etiquetas que cambian entre años,"RemoteWork, Employment y Age con etiquetas distintas; Currency con 2 formatos; 'ASP.NET Core ' / 'ASP.NET CORE' / 'ASP.NET Core', 'React.js' / 'React'",Al unir los años se crean categorías falsas o duplicadas,Armonizar con un diccionario de equivalencias y str.strip()
8,Encuesta 2025,Más nulos en tecnologías y país,LanguageHaveWorkedWith nulo 35.6 % y Country 28.0 % en 2025,Menos casos por tecnología y región en el último año,Excluir esas filas del análisis por tecnología o región y reportar el tamaño de muestra por año
9,Encuestas vs. PIB,Nombres de país no coinciden,"~30 países sin coincidencia con OWID (p. ej. 'United States of America', 'United Kingdom of Great Britain and Northern Ireland', 'Russian Federation')",Al unir se perdería cerca del 30 % de las respuestas,Crear una tabla de equivalencias de países o unir por código ISO3; excluir 'Nomadic'


## 5. Conclusiones

1. **Los datos sirven para la pregunta, pero no se pueden usar en bruto.** Las cuatro encuestas tienen salario, tecnologías, experiencia y país para decenas de miles de desarrolladores profesionales. Las dos fuentes de PIB cubren 2022–2025.
2. **El salario es el punto más delicado:** tiene muchos nulos y valores extremos. El análisis debe trabajar solo con respuestas con salario, recortar atípicos y usar la mediana.
3. **Los cuatro años no son directamente comparables:** cambian las categorías de empleo, la variable de experiencia profesional (2025 no tiene `YearsCodePro`), los nombres de tecnologías y el formato de algunas columnas. Antes de unirlos hay que armonizarlos.
4. **Para la dimensión regional** hay que construir una tabla de equivalencias de países, filtrar los agregados del PIB y elegir una sola unidad de PIB.
5. **Siguiente paso:** un cuaderno de limpieza que aplique las acciones de la tabla de hallazgos y genere en `data/processed/` un dataset unificado de 2022–2025 con la población de interés.

El informe completo está en `reports/informe_diagnostico.md`.